# Strategy Tester (No TBM / CPCV / HMM / XGBoost)

Goal:
1. Compare multiple raw strategy families under one unified backtester.
2. Include realistic friction checks (spread cap, slippage).
3. Rank by robustness (plateau stability) and profit-per-trade.
4. Export top candidates for later wiring into GoldRegimeX_Explorer.

In [1]:
# Imports + Config
import os
import math
import json
import time
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from joblib import Parallel, delayed
    JOBLIB_OK = True
except Exception:
    JOBLIB_OK = False

np.random.seed(42)

# -----------------------------
# Runtime controls
# -----------------------------
N_JOBS = 6  # lower to reduce memory pressure under loky
QUICK_MODE = False
RESEARCH_YEARS = 5  # enforced always, regardless of QUICK_MODE

# -----------------------------
# Data paths
# -----------------------------
M5_PATH = Path("data/raw/XAU_5m_data.csv")
M15_PATH = Path("data/raw/XAU_15m_data.csv")

# -----------------------------
# Core assumptions
# -----------------------------
TIMEFRAMES = ["M15", "M5"]
INITIAL_BALANCE_CENTS = 1500.0
PIP_SIZE_PRICE = 0.10
PIP_VALUE_CENTS_PER_1LOT = 100.0
SLIPPAGE_PIPS = 0.30
SPREAD_CAP_POINTS = 40.0
COMMISSION_CENTS_PER_TRADE = 0.0

# Split position support
POSITION_A = 0.01
POSITION_B = 0.01

# M5 Grids (Tighter targets, wider stops to survive noise and high friction)
M5_ADX_GRID = [20, 25]
M5_PULLBACK_RSI_GRID = [30, 35]
M5_CONFIRMATION_GRID = [1, 2]
M5_ATR_STOP_GRID = [1.0, 1.5, 2.0, 2.5, 3.0]  # Extended upward to allow M5 breathing room
M5_LEG_A_TARGET_GRID = [0.8, 1.0]
M5_ENTRY_TARGET_GRID = [1.0, 1.5]

# Exit modes
EXIT_MODELS = [
    "fixed_tp",
    "mr_exit",
    "fixed_tp_plus_mr",
    "partial_tp_plus_mr",
    "partial_tp_mr_time_stop",
]

# ATR multiple grids
LEG_A_ATR_TARGET_GRID = [1.0, 1.5, 2.0]  # Leg A target (ATR multiple)
ENTRY_ATR_TARGET_GRID = [2.0, 2.5, 3.0]  # Leg B (fixed TP) ATR multiple
ATR_TARGET_GRID = ENTRY_ATR_TARGET_GRID  # keep existing name used by strategies

# Leg C constants (Scale-in removed from grid search to save compute)
LEG_C_ATR_TARGET = 0.5
LEG_C_ATR_STOP = 0.5

TIME_STOP_GRID_BY_TF = {
    "M15": [120, 180, 240],
    "M5": [30, 60, 90],
}
TRAIL_MULT_GRID = [1.5, 2.0, 2.5]

# Session filter options
SESSION_FILTER_VALUES = [None, "London", "NY", "London_NY"]

# -----------------------------
# Strategy A grids (Loosened heavily to restore M15 trade frequency)
# -----------------------------
# Lowered ADX so we don't need a massive macro trend to trigger
ADX_GRID = [15.0, 18.0, 20.0, 25.0] 
# Loosened RSI heavily. On a 15-minute chart, trends rarely retrace all the way to 25/30 RSI. 
# Allowing 35-45 RSI captures shallower, highly valid trend pullbacks.
PULLBACK_RSI_GRID = [25, 30, 35.0, 40.0, 45.0] 
CONFIRMATION_GRID = [1, 2]  # Get in faster, less lag
ATR_STOP_GRID = [0.8, 1.0, 1.5, 2.0, 2.5, 3.0]  # Loosened to allow more breathing room for M15 noise

# -----------------------------
# Strategy B grids (Loosened for Frequency)
# -----------------------------
ATR_EXPANSION_GRID = [1.1, 1.25, 1.4]  # 1.5+ is too rare; 1.1+ catches standard volatility cycles
BREAKOUT_LOOKBACK_GRID = [10, 20, 30]
BREAKOUT_BUFFER_GRID = [0.0, 0.25]

if QUICK_MODE:
    # Stronger quick cut to keep runtime practical
    ADX_GRID = ADX_GRID[:2]
    PULLBACK_RSI_GRID = PULLBACK_RSI_GRID[:2]
    CONFIRMATION_GRID = CONFIRMATION_GRID[:2]
    ATR_STOP_GRID = ATR_STOP_GRID[:1]
    LEG_A_ATR_TARGET_GRID = LEG_A_ATR_TARGET_GRID[:2]
    ENTRY_ATR_TARGET_GRID = ENTRY_ATR_TARGET_GRID[:2]
    ATR_TARGET_GRID = ENTRY_ATR_TARGET_GRID

    ATR_EXPANSION_GRID = ATR_EXPANSION_GRID[:3]
    BREAKOUT_LOOKBACK_GRID = BREAKOUT_LOOKBACK_GRID[:2]
    BREAKOUT_BUFFER_GRID = BREAKOUT_BUFFER_GRID[:2]

    SESSION_FILTER_VALUES = [None, "London_NY"]
    EXIT_MODELS = ["fixed_tp", "mr_exit", "fixed_tp_plus_mr"]

    TIME_STOP_GRID_BY_TF["M15"] = TIME_STOP_GRID_BY_TF["M15"][:2]
    TIME_STOP_GRID_BY_TF["M5"] = TIME_STOP_GRID_BY_TF["M5"][:2]
    TRAIL_MULT_GRID = TRAIL_MULT_GRID[:2]

print("JOBLIB_OK:", JOBLIB_OK)
print("N_JOBS:", N_JOBS)
print("QUICK_MODE:", QUICK_MODE)
print("RESEARCH_YEARS (enforced):", RESEARCH_YEARS)

JOBLIB_OK: True
N_JOBS: 6
QUICK_MODE: False
RESEARCH_YEARS (enforced): 5


In [2]:
# Optional exploratory speed cap (applies to QUICK_MODE True/False)
ENABLE_ENTRY_CAP = False

# Hard cap per (timeframe, strategy, exit_model) on ENTRY parameter combos.
# This is the biggest speed lever for QUICK_MODE=False.
if QUICK_MODE:
    ENTRY_CAP_BY_TF = {"M15": 80, "M5": 40}
else:
    ENTRY_CAP_BY_TF = {"M15": 200, "M5": 80}

# Seed for deterministic subset selection
ENTRY_CAP_SEED_BASE = 42

print("ENABLE_ENTRY_CAP:", ENABLE_ENTRY_CAP)
print("ENTRY_CAP_BY_TF:", ENTRY_CAP_BY_TF)
print("ENTRY_CAP_SEED_BASE:", ENTRY_CAP_SEED_BASE)

ENABLE_ENTRY_CAP: False
ENTRY_CAP_BY_TF: {'M15': 200, 'M5': 80}
ENTRY_CAP_SEED_BASE: 42


In [3]:
# Data loading, strict 5-year reduction, indicators, and rule-based regimes

def resolve_path(path: Path) -> Path:
    if path.exists():
        return path
    alt = Path.cwd().parent / path
    if alt.exists():
        return alt
    raise FileNotFoundError(f"Path not found: {path}")

def _normalize_ohlc(df: pd.DataFrame) -> pd.DataFrame:
    cols = {c.lower(): c for c in df.columns}
    rename = {}
    for key in ["open", "high", "low", "close", "volume", "spread"]:
        if key in cols:
            rename[cols[key]] = key
    out = df.rename(columns=rename)
    missing = [c for c in ["open", "high", "low", "close"] if c not in out.columns]
    if missing:
        raise ValueError(f"Missing OHLC columns: {missing}")
    return out

def read_xau_raw(path: Path) -> pd.DataFrame:
    path = resolve_path(path)
    df = pd.read_csv(path, sep=";")
    if "Date" not in df.columns:
        raise ValueError(f"Date column missing in {path}")
    df["Date"] = pd.to_datetime(df["Date"], format="%Y.%m.%d %H:%M", errors="coerce")
    df = df.dropna(subset=["Date"]).set_index("Date").sort_index()
    df = _normalize_ohlc(df)
    if "spread" not in df.columns:
        df["spread"] = SPREAD_CAP_POINTS
    return df

def load_recent_years(df: pd.DataFrame, years: int) -> pd.DataFrame:
    last_date = df.index.max()
    start_date = last_date - pd.DateOffset(years=int(years))
    return df.loc[df.index >= start_date].copy()

def enforce_recent_window(df_full: pd.DataFrame, df_trim: pd.DataFrame, years: int, label: str):
    last_date = df_full.index.max()
    target_start = last_date - pd.DateOffset(years=int(years))
    observed_start = df_trim.index.min()
    if observed_start < target_start:
        raise RuntimeError(f"{label} window enforcement failed: observed_start={observed_start}, target_start={target_start}")
    print(f"{label} enforced window: {observed_start} -> {last_date}")

def ema(series: pd.Series, period: int) -> pd.Series:
    return series.ewm(span=int(period), adjust=False, min_periods=int(period)).mean()

def true_range(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    prev_close = close.shift(1)
    return pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)

def atr(high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14) -> pd.Series:
    tr = true_range(high, low, close)
    return tr.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean()

def rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return (100.0 - (100.0 / (1.0 + rs))).fillna(50.0)

def adx(high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14) -> pd.Series:
    up_move = high.diff()
    down_move = -low.diff()
    plus_dm = up_move.where((up_move > down_move) & (up_move > 0.0), 0.0)
    minus_dm = down_move.where((down_move > up_move) & (down_move > 0.0), 0.0)
    atr_v = atr(high, low, close, period=period).replace(0.0, np.nan)
    plus_di = 100.0 * plus_dm.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean() / atr_v
    minus_di = 100.0 * minus_dm.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean() / atr_v
    dx = 100.0 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0.0, np.nan)
    return dx.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean().fillna(0.0)

def add_session_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    hour = out.index.hour
    london = (hour >= 7) & (hour < 16)
    ny = (hour >= 13) & (hour < 21)
    overlap = (hour >= 13) & (hour < 16)
    out["session"] = np.where(overlap, "OVERLAP", np.where(london, "LONDON", np.where(ny, "NEW_YORK", "ASIA")))
    out["session_mask_none"] = True
    out["session_mask_london"] = london
    out["session_mask_ny"] = ny
    out["session_mask_london_ny"] = london | ny
    return out

def build_features(exec_tf: str, m5_df: pd.DataFrame, m15_df: pd.DataFrame) -> pd.DataFrame:
    exec_tf = exec_tf.upper()
    if exec_tf not in ("M5", "M15"):
        raise ValueError(f"Unsupported timeframe: {exec_tf}")

    exec_df = m5_df.copy() if exec_tf == "M5" else m15_df.copy()
    trend_df = m15_df.copy()

    exec_df["rsi5"] = rsi(exec_df["close"], period=5)
    exec_df["atr14"] = atr(exec_df["high"], exec_df["low"], exec_df["close"], period=14)
    exec_df["atr100"] = atr(exec_df["high"], exec_df["low"], exec_df["close"], period=100)
    exec_df["atr_expansion"] = exec_df["atr14"] / exec_df["atr100"].replace(0.0, np.nan)

    for lb in sorted(set(BREAKOUT_LOOKBACK_GRID)):
        exec_df[f"roll_high_{lb}"] = exec_df["high"].rolling(lb, min_periods=lb).max().shift(1)
        exec_df[f"roll_low_{lb}"] = exec_df["low"].rolling(lb, min_periods=lb).min().shift(1)

    trend_df["m15_ema50"] = ema(trend_df["close"], period=50)
    trend_df["m15_ema200"] = ema(trend_df["close"], period=200)
    trend_df["m15_adx14"] = adx(trend_df["high"], trend_df["low"], trend_df["close"], period=14)

    if exec_tf == "M5":
        ex = exec_df.reset_index().rename(columns={exec_df.index.name or "index": "time"})
        tr = trend_df.reset_index().rename(columns={trend_df.index.name or "index": "time"})
        merged = pd.merge_asof(
            ex.sort_values("time"),
            tr[["time", "m15_ema50", "m15_ema200", "m15_adx14"]].sort_values("time"),
            on="time",
            direction="backward",
        ).set_index("time")
    else:
        merged = exec_df.copy()
        merged["m15_ema50"] = trend_df["m15_ema50"].reindex(merged.index)
        merged["m15_ema200"] = trend_df["m15_ema200"].reindex(merged.index)
        merged["m15_adx14"] = trend_df["m15_adx14"].reindex(merged.index)

    merged = add_session_features(merged)
    merged["spread"] = merged["spread"].fillna(SPREAD_CAP_POINTS)

        # Rule-based regime classification
    is_trend = (merged["m15_adx14"] > 15.0) & (merged["atr_expansion"] < 1.3)
    is_shock = merged["atr_expansion"] >= 1.3

    merged["regime_str"] = np.where(is_shock, "SHOCK", np.where(is_trend, "TREND", "MR"))
    merged["regime_code"] = np.where(is_shock, 2, np.where(is_trend, 1, 3)).astype(np.int32)
    
    required = [
        "open", "high", "low", "close", "spread",
        "rsi5", "atr14", "atr100", "atr_expansion",
        "m15_ema50", "m15_ema200", "m15_adx14",
        "session", "regime_str", "regime_code",
        "session_mask_none", "session_mask_london", "session_mask_ny", "session_mask_london_ny",
    ]
    merged = merged.dropna(subset=[c for c in required if c in merged.columns]).copy()
    return merged

m5_raw_full = read_xau_raw(M5_PATH)
m15_raw_full = read_xau_raw(M15_PATH)

m5_raw = load_recent_years(m5_raw_full, years=RESEARCH_YEARS)
m15_raw = load_recent_years(m15_raw_full, years=RESEARCH_YEARS)

enforce_recent_window(m5_raw_full, m5_raw, years=RESEARCH_YEARS, label="M5")
enforce_recent_window(m15_raw_full, m15_raw, years=RESEARCH_YEARS, label="M15")

FEATURES_BY_TF = {
    "M5": build_features("M5", m5_raw, m15_raw),
    "M15": build_features("M15", m5_raw, m15_raw),
}

print("M5 full rows:", len(m5_raw_full), "| recent rows:", len(m5_raw))
print("M15 full rows:", len(m15_raw_full), "| recent rows:", len(m15_raw))
for tf in TIMEFRAMES:
    print(tf, "feature rows:", len(FEATURES_BY_TF[tf]))
display(FEATURES_BY_TF["M5"].head(3))

M5 enforced window: 2021-05-31 01:05:00 -> 2026-05-29 23:50:00
M15 enforced window: 2021-05-31 01:00:00 -> 2026-05-29 23:30:00
M5 full rows: 1468386 | recent rows: 347571
M15 full rows: 502563 | recent rows: 116101
M15 feature rows: 115902
M5 feature rows: 346980


,open,high,low,close,volume,spread,rsi5,atr14,atr100,atr_expansion,...,m15_ema50,m15_ema200,m15_adx14,session,session_mask_none,session_mask_london,session_mask_ny,session_mask_london_ny,regime_str,regime_code
time,,,,,,,,,,,,,,,,,,,,,
2021-06-02 09:00:00,1898.52,1898.52,1895.16,1895.64,944,40.0,26.791187,1.146459,1.004396,1.141442,...,1899.620462,1903.612551,17.554024,LONDON,True,True,False,True,TREND,1
2021-06-02 09:05:00,1895.62,1897.71,1895.58,1897.14,600,40.0,44.362487,1.216712,1.015652,1.197962,...,1899.620462,1903.612551,17.554024,LONDON,True,True,False,True,TREND,1
2021-06-02 09:10:00,1897.14,1898.45,1896.97,1898.02,549,40.0,52.689668,1.235519,1.020295,1.210942,...,1899.620462,1903.612551,17.554024,LONDON,True,True,False,True,TREND,1


In [4]:
# Legacy cell disabled intentionally.
# Data loading and feature building are now centralized in the previous cell
# and strictly enforced to RESEARCH_YEARS only.

print("Legacy research-frame cell disabled. Using FEATURES_BY_TF from strict recent-window pipeline.")

Legacy research-frame cell disabled. Using FEATURES_BY_TF from strict recent-window pipeline.


In [5]:
# Cell 5: Strategy Definitions & Regime-Based Signal Router (With Macro Filter)
class BaseStrategy:
    name = "base"
    param_grid = {}
    param_cols = []

    def iter_param_dicts(self):
        keys = list(self.param_cols)
        vals = [self.param_grid[k] for k in keys]
        for combo in itertools.product(*vals):
            yield dict(zip(keys, combo))

    def generate_signals(self, df: pd.DataFrame, params: dict) -> pd.Series:
        raise NotImplementedError

def session_col_from_value(session_filter):
    if session_filter is None:
        return "session_mask_none"
    s = str(session_filter).lower()
    if s == "london":
        return "session_mask_london"
    if s == "ny":
        return "session_mask_ny"
    if s == "london_ny":
        return "session_mask_london_ny"
    raise ValueError(f"Unsupported session_filter: {session_filter}")

class TrendPullbackStrategy(BaseStrategy):
    name = "trend_pullback"
    param_grid = {
        "adx_threshold": ADX_GRID,
        "pullback_rsi": PULLBACK_RSI_GRID,
        "confirmation_bars": CONFIRMATION_GRID,
        "atr_stop": ATR_STOP_GRID,
        "atr_target": ATR_TARGET_GRID,
        "session_filter": SESSION_FILTER_VALUES,
    }
    param_cols = list(param_grid.keys())

    def generate_signals(self, df: pd.DataFrame, params: dict) -> pd.Series:
        adx_threshold = float(params["adx_threshold"])
        pullback_rsi = float(params["pullback_rsi"])
        confirmation_bars = int(params["confirmation_bars"])
        session_col = session_col_from_value(params["session_filter"])

        trend_up_raw = (df["m15_ema50"] > df["m15_ema200"]) & (df["m15_adx14"] > adx_threshold)
        trend_dn_raw = (df["m15_ema50"] < df["m15_ema200"]) & (df["m15_adx14"] > adx_threshold)

        if confirmation_bars > 1:
            trend_up = trend_up_raw.rolling(confirmation_bars, min_periods=confirmation_bars).sum().eq(confirmation_bars)
            trend_dn = trend_dn_raw.rolling(confirmation_bars, min_periods=confirmation_bars).sum().eq(confirmation_bars)
        else:
            trend_up, trend_dn = trend_up_raw, trend_dn_raw

        long_cond = trend_up & (df["rsi5"] < pullback_rsi) & df[session_col].astype(bool)
        short_cond = trend_dn & (df["rsi5"] > (100.0 - pullback_rsi)) & df[session_col].astype(bool)

        sig = pd.Series(0, index=df.index, dtype=np.int8)
        sig.loc[long_cond.fillna(False)] = 1
        sig.loc[short_cond.fillna(False)] = -1
        return sig

class VolatilityExpansionStrategy(BaseStrategy):
    name = "volatility_expansion"
    param_grid = {
        "atr_expansion_threshold": ATR_EXPANSION_GRID,
        "breakout_lookback": BREAKOUT_LOOKBACK_GRID,
        "breakout_buffer": BREAKOUT_BUFFER_GRID,
        "atr_stop": ATR_STOP_GRID,
        "atr_target": ATR_TARGET_GRID,
        "session_filter": SESSION_FILTER_VALUES,
    }
    param_cols = list(param_grid.keys())

    def generate_signals(self, df: pd.DataFrame, params: dict) -> pd.Series:
        thr = float(params["atr_expansion_threshold"])
        lb = int(params["breakout_lookback"])
        buf_mult = float(params["breakout_buffer"])
        session_col = session_col_from_value(params["session_filter"])

        high_col = f"roll_high_{lb}"
        low_col = f"roll_low_{lb}"
        breakout_buffer = buf_mult * df["atr14"]
        is_expansion = df["atr_expansion"] > thr

        # MACRO TREND FILTER: Ensures entries align with higher timeframe momentum
        macro_up = df["m15_ema50"] > df["m15_ema200"]
        macro_dn = df["m15_ema50"] < df["m15_ema200"]

        long_cond = is_expansion & (df["close"] > (df[high_col] + breakout_buffer)) & macro_up & df[session_col].astype(bool)
        short_cond = is_expansion & (df["close"] < (df[low_col] - breakout_buffer)) & macro_dn & df[session_col].astype(bool)

        sig = pd.Series(0, index=df.index, dtype=np.int8)
        sig.loc[long_cond.fillna(False)] = 1
        sig.loc[short_cond.fillna(False)] = -1
        return sig

STRATEGIES = {
    "trend_pullback": TrendPullbackStrategy(),
    "volatility_expansion": VolatilityExpansionStrategy(),
}

def generate_routed_signals(df: pd.DataFrame, params: dict, strategy_name: str) -> pd.Series:
    raw_signals = STRATEGIES[strategy_name].generate_signals(df, params)
    routed = pd.Series(0, index=df.index, dtype=np.int8)
    if strategy_name == "trend_pullback":
        mask = df["regime_code"] == 1
        routed.loc[mask] = raw_signals.loc[mask]
    elif strategy_name == "volatility_expansion":
        mask = df["regime_code"] == 2
        routed.loc[mask] = raw_signals.loc[mask]
    return routed

In [6]:
# Cell 6: Numba Backtest Engine with Early Eject, Pyramiding (Scale-In), and Asymmetric Guard

from numba import njit
import numpy as np
import pandas as pd
import math

def compute_metrics(trades_df: pd.DataFrame, equity_curve: list[float], initial_balance: float, ending_balance: float) -> dict:
    if trades_df.empty:
        return {
            "profit_factor": 0.0, "sharpe": 0.0, "sortino": 0.0, "calmar": 0.0, 
            "max_drawdown": 0.0, "expectancy": 0.0, "win_rate": 0.0, "trade_count": 0, 
            "net_profit": float(ending_balance - initial_balance), "profit_per_trade": 0.0, "net_return_pct": 0.0
        }

    pnl = trades_df["pnl_cents"].to_numpy(dtype=float)
    trade_count = int(len(pnl))
    net_profit = float(np.sum(pnl))
    gross_profit = float(np.sum(pnl[pnl > 0]))
    gross_loss = float(-np.sum(pnl[pnl < 0]))
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else (10.0 if gross_profit > 0 else 0.0)
    win_rate = float(np.mean(pnl > 0))
    expectancy = float(np.mean(pnl))
    profit_per_trade = float(net_profit / max(trade_count, 1))

    r = pnl / max(initial_balance, 1e-9)
    r_std = float(np.std(r, ddof=0))
    sharpe = float((np.mean(r) / r_std) * math.sqrt(len(r))) if r_std > 0 else 0.0

    downside = r[r < 0]
    d_std = float(np.std(downside, ddof=0)) if len(downside) > 0 else 0.0
    sortino = float((np.mean(r) / d_std) * math.sqrt(len(r))) if d_std > 0 else 0.0

    eq = np.array(equity_curve, dtype=float)
    
    # RISK MANAGEMENT: Drawdown calculations based on an average of multiple positions
    window = min(3, len(eq))
    avg_eq = pd.Series(eq).rolling(window=window, min_periods=1).mean().to_numpy()
    peaks = np.maximum.accumulate(avg_eq)
    dd = peaks - avg_eq
    max_dd_pct = float(np.max(dd / np.maximum(peaks, 1e-9)) * 100.0) if len(dd) > 0 else 0.0
    
    net_return_pct = float((ending_balance / initial_balance - 1.0) * 100.0)
    calmar = float(net_return_pct / max_dd_pct) if max_dd_pct > 0 else 0.0

    return {
        "profit_factor": profit_factor, "sharpe": sharpe, "sortino": sortino, "calmar": calmar, 
        "max_drawdown": max_dd_pct, "expectancy": expectancy, "win_rate": win_rate, "trade_count": trade_count, 
        "net_profit": net_profit, "profit_per_trade": profit_per_trade, "net_return_pct": net_return_pct
    }

@njit(cache=True)
def _run_backtest_numba(
    sig, high, low, close, spread, atr14, regime_code, time_minutes, 
    entry_atr_stop, entry_atr_target, leg_a_atr_target, exit_mode_code, 
    time_stop_minutes, trail_mult, initial_balance, pip_size_price, 
    pip_value_per_1lot, slippage_pips, spread_cap_points, commission_cents, is_m5
):
    n = sig.shape[0]
    max_trades = n * 3

    trade_entry_idx = np.empty(max_trades, dtype=np.int64)
    trade_exit_idx = np.empty(max_trades, dtype=np.int64)
    trade_leg = np.empty(max_trades, dtype=np.int8)
    trade_side = np.empty(max_trades, dtype=np.int8)
    trade_entry_px = np.empty(max_trades, dtype=np.float64)
    trade_exit_px = np.empty(max_trades, dtype=np.float64)
    trade_move_pips = np.empty(max_trades, dtype=np.float64)
    trade_pnl_cents = np.empty(max_trades, dtype=np.float64)
    trade_reason = np.empty(max_trades, dtype=np.int8)
    trade_entry_regime = np.empty(max_trades, dtype=np.int32)

    eq = np.empty(max_trades + 1, dtype=np.float64)
    eq[0] = initial_balance
    eq_count = 1

    legs = np.zeros((3, 7), dtype=np.float64)
    trade_count = 0
    balance = initial_balance
    active_trade_regime = 0
    leg_a_profit_hit = 0
    leg_b_profit_hit = 0

    enable_fixed_tp = exit_mode_code in (0, 2)
    enable_mr = exit_mode_code in (1, 2, 3, 4)
    enable_time_stop = exit_mode_code == 4
    enable_trail = exit_mode_code == 4

    for i in range(n):
        s = int(sig[i])
        current_regime = int(regime_code[i])

        is_flat = (legs[0, 0] == 0.0 and legs[1, 0] == 0.0 and legs[2, 0] == 0.0)
        leg_a_closed_b_open = (legs[0, 0] == 0.0 and legs[1, 0] == 1.0 and leg_a_profit_hit == 1)
        leg_b_closed_a_open = (legs[1, 0] == 0.0 and legs[0, 0] == 1.0 and leg_b_profit_hit == 1)
        can_scale_in = (legs[2, 0] == 0.0 and (leg_a_closed_b_open or leg_b_closed_a_open))

        # ENTRY & SCALE-IN LOGIC
        if (is_flat or can_scale_in) and s != 0:
            sp_points = spread[i]
            atr_now = atr14[i]
            
            effective_spread = (sp_points / 10.0) * pip_size_price
            if np.isfinite(atr_now) and atr_now > 0.0 and effective_spread <= (0.8 * atr_now):
                side = 1 if s > 0 else -1

                if can_scale_in:
                    runner_idx = 1 if leg_a_closed_b_open else 0
                    if side != int(legs[runner_idx, 2]):
                        continue 

                spread_price = effective_spread
                slippage_price = slippage_pips * pip_size_price
                close_px = close[i]

                # RISK MANAGEMENT: Dynamic Asymmetric Guard Factor
                if is_m5 == 1:
                    guard_factor = 0.85 if side == 1 else 0.75
                else:
                    guard_factor = 0.65

                if is_flat:
                    leg_a_profit_hit = 0
                    leg_b_profit_hit = 0
                    
                    intended_stop_dist = entry_atr_stop * atr_now
                    actual_stop_dist = intended_stop_dist * guard_factor
                    tp_dist = entry_atr_target * atr_now

                    if side == 1:
                        entry_px = close_px + spread_price + slippage_price
                        stop_px = entry_px - actual_stop_dist
                        fixed_tp_px = entry_px + tp_dist
                        leg_a_tp_px = entry_px + (leg_a_atr_target * atr_now) if leg_a_atr_target > 0.0 else np.nan
                    else:
                        entry_px = close_px - slippage_price
                        stop_px = entry_px + actual_stop_dist + spread_price
                        fixed_tp_px = entry_px - tp_dist
                        leg_a_tp_px = entry_px - (leg_a_atr_target * atr_now) if leg_a_atr_target > 0.0 else np.nan

                    active_trade_regime = current_regime
                    legs[0, 0], legs[0, 1], legs[0, 2], legs[0, 3], legs[0, 4], legs[0, 5], legs[0, 6] = 1.0, POSITION_A, side, entry_px, stop_px, leg_a_tp_px, i
                    legs[1, 0], legs[1, 1], legs[1, 2], legs[1, 3], legs[1, 4], legs[1, 5], legs[1, 6] = 1.0, POSITION_B, side, entry_px, stop_px, fixed_tp_px if enable_fixed_tp else np.nan, i

                elif can_scale_in:
                    leg_c_lot = POSITION_A if leg_a_closed_b_open else POSITION_B
                    
                    tighter_stop = 0.5 * atr_now
                    actual_stop_dist = tighter_stop * guard_factor
                    tighter_tp = 0.5 * atr_now

                    if side == 1:
                        entry_px = close_px + spread_price + slippage_price
                        stop_px = entry_px - actual_stop_dist
                        fixed_tp_px = entry_px + tighter_tp
                    else:
                        entry_px = close_px - slippage_price
                        stop_px = entry_px + actual_stop_dist + spread_price
                        fixed_tp_px = entry_px - tighter_tp

                    legs[2, 0], legs[2, 1], legs[2, 2], legs[2, 3], legs[2, 4], legs[2, 5], legs[2, 6] = 1.0, leg_c_lot, side, entry_px, stop_px, fixed_tp_px, i

        bar_high, bar_low, bar_close, atr_now = high[i], low[i], close[i], atr14[i]

        # EXIT PROCESSING
        for leg_idx in range(3):
            if legs[leg_idx, 0] == 0.0:
                continue

            leg_side = int(legs[leg_idx, 2])
            leg_entry = legs[leg_idx, 3]
            leg_stop = legs[leg_idx, 4]
            leg_tp = legs[leg_idx, 5]
            leg_lot = legs[leg_idx, 1]
            leg_entry_i = int(legs[leg_idx, 6])

            if enable_trail and leg_idx == 1 and np.isfinite(atr_now) and atr_now > 0.0:
                dist = trail_mult * atr_now
                if leg_side == 1:
                    if bar_close - dist > leg_stop:
                        leg_stop = bar_close - dist
                else:
                    if bar_close + dist < leg_stop:
                        leg_stop = bar_close + dist
                legs[leg_idx, 4] = leg_stop

            # 1. Stop Loss Hit
            stop_hit = (bar_low <= leg_stop) if leg_side == 1 else (bar_high >= leg_stop)
            if stop_hit:
                exit_px = leg_stop
                move_pips = ((exit_px - leg_entry) / pip_size_price) * leg_side
                pnl_cents = move_pips * (leg_lot * pip_value_per_1lot) - commission_cents
                trade_entry_idx[trade_count] = leg_entry_i
                trade_exit_idx[trade_count] = i
                trade_leg[trade_count] = leg_idx
                trade_side[trade_count] = leg_side
                trade_entry_px[trade_count] = leg_entry
                trade_exit_px[trade_count] = exit_px
                trade_move_pips[trade_count] = move_pips
                trade_pnl_cents[trade_count] = pnl_cents
                trade_reason[trade_count] = 1
                trade_entry_regime[trade_count] = active_trade_regime
                trade_count += 1
                balance += pnl_cents
                eq[eq_count] = balance
                eq_count += 1
                legs[leg_idx, 0] = 0.0
                if leg_idx == 1: leg_a_profit_hit = 0
                continue

            # 2. Leg 0 (A) Profit Target
            if leg_idx == 0 and np.isfinite(leg_tp):
                tp_hit = (bar_high >= leg_tp) if leg_side == 1 else (bar_low <= leg_tp)
                if tp_hit:
                    exit_px = leg_tp
                    move_pips = ((exit_px - leg_entry) / pip_size_price) * leg_side
                    pnl_cents = move_pips * (leg_lot * pip_value_per_1lot) - commission_cents
                    trade_entry_idx[trade_count] = leg_entry_i
                    trade_exit_idx[trade_count] = i
                    trade_leg[trade_count] = leg_idx
                    trade_side[trade_count] = leg_side
                    trade_entry_px[trade_count] = leg_entry
                    trade_exit_px[trade_count] = exit_px
                    trade_move_pips[trade_count] = move_pips
                    trade_pnl_cents[trade_count] = pnl_cents
                    trade_reason[trade_count] = 2
                    trade_entry_regime[trade_count] = active_trade_regime
                    trade_count += 1
                    balance += pnl_cents
                    eq[eq_count] = balance
                    eq_count += 1
                    legs[leg_idx, 0] = 0.0
                    leg_a_profit_hit = 1
                    continue

            # 3. Leg B or Leg C Fixed TP
            if leg_idx in (1, 2) and enable_fixed_tp and np.isfinite(leg_tp):
                tp_hit = (bar_high >= leg_tp) if leg_side == 1 else (bar_low <= leg_tp)
                if tp_hit:
                    exit_px = leg_tp
                    move_pips = ((exit_px - leg_entry) / pip_size_price) * leg_side
                    pnl_cents = move_pips * (leg_lot * pip_value_per_1lot) - commission_cents
                    trade_entry_idx[trade_count] = leg_entry_i
                    trade_exit_idx[trade_count] = i
                    trade_leg[trade_count] = leg_idx
                    trade_side[trade_count] = leg_side
                    trade_entry_px[trade_count] = leg_entry
                    trade_exit_px[trade_count] = exit_px
                    trade_move_pips[trade_count] = move_pips
                    trade_pnl_cents[trade_count] = pnl_cents
                    trade_reason[trade_count] = 3
                    trade_entry_regime[trade_count] = active_trade_regime
                    trade_count += 1
                    balance += pnl_cents
                    eq[eq_count] = balance
                    eq_count += 1
                    legs[leg_idx, 0] = 0.0
                    if leg_idx == 1:
                        leg_b_profit_hit = 1
                    continue

            # 4. Structural MR Exit (Clears all active legs)
            if leg_idx == 1 and enable_mr and current_regime == 3:
                for l_id in range(3):
                    if legs[l_id, 0] == 1.0:
                        m_pips = ((bar_close - legs[l_id, 3]) / pip_size_price) * int(legs[l_id, 2])
                        p_cents = m_pips * (legs[l_id, 1] * pip_value_per_1lot) - commission_cents
                        trade_entry_idx[trade_count] = int(legs[l_id, 6])
                        trade_exit_idx[trade_count] = i
                        trade_leg[trade_count] = l_id
                        trade_side[trade_count] = int(legs[l_id, 2])
                        trade_entry_px[trade_count] = legs[l_id, 3]
                        trade_exit_px[trade_count] = bar_close
                        trade_move_pips[trade_count] = m_pips
                        trade_pnl_cents[trade_count] = p_cents
                        trade_reason[trade_count] = 4
                        trade_entry_regime[trade_count] = active_trade_regime
                        trade_count += 1
                        balance += p_cents
                        legs[l_id, 0] = 0.0
                eq[eq_count] = balance
                eq_count += 1
                leg_a_profit_hit = 0
                leg_b_profit_hit = 0
                break

            # 5. Dynamic Time Stop (Clears all active legs)
            if leg_idx == 1 and enable_time_stop and time_stop_minutes > 0.0:
                if (time_minutes[i] - time_minutes[leg_entry_i]) >= time_stop_minutes:
                    for l_id in range(3):
                        if legs[l_id, 0] == 1.0:
                            m_pips = ((bar_close - legs[l_id, 3]) / pip_size_price) * int(legs[l_id, 2])
                            p_cents = m_pips * (legs[l_id, 1] * pip_value_per_1lot) - commission_cents
                            trade_entry_idx[trade_count] = int(legs[l_id, 6])
                            trade_exit_idx[trade_count] = i
                            trade_leg[trade_count] = l_id
                            trade_side[trade_count] = int(legs[l_id, 2])
                            trade_entry_px[trade_count] = legs[l_id, 3]
                            trade_exit_px[trade_count] = bar_close
                            trade_move_pips[trade_count] = m_pips
                            trade_pnl_cents[trade_count] = p_cents
                            trade_reason[trade_count] = 5
                            trade_entry_regime[trade_count] = active_trade_regime
                            trade_count += 1
                            balance += p_cents
                            legs[l_id, 0] = 0.0
                    eq[eq_count] = balance
                    eq_count += 1
                    leg_a_profit_hit = 0
                    leg_b_profit_hit = 0
                    break

    # EOD Cleanup
    if legs[0, 0] == 1.0 or legs[1, 0] == 1.0 or legs[2, 0] == 1.0:
        last_i = n - 1
        for leg_idx in range(3):
            if legs[leg_idx, 0] == 1.0:
                m_pips = ((close[last_i] - legs[leg_idx, 3]) / pip_size_price) * int(legs[leg_idx, 2])
                p_cents = m_pips * (legs[leg_idx, 1] * pip_value_per_1lot) - commission_cents
                trade_entry_idx[trade_count] = int(legs[leg_idx, 6])
                trade_exit_idx[trade_count] = last_i
                trade_leg[trade_count] = leg_idx
                trade_side[trade_count] = int(legs[leg_idx, 2])
                trade_entry_px[trade_count] = legs[leg_idx, 3]
                trade_exit_px[trade_count] = close[last_i]
                trade_move_pips[trade_count] = m_pips
                trade_pnl_cents[trade_count] = p_cents
                trade_reason[trade_count] = 6
                trade_entry_regime[trade_count] = active_trade_regime
                trade_count += 1
                balance += p_cents
        eq[eq_count] = balance
        eq_count += 1

    return (
        trade_entry_idx, trade_exit_idx, trade_leg, trade_side, trade_entry_px,
        trade_exit_px, trade_move_pips, trade_pnl_cents, trade_reason,
        trade_entry_regime, trade_count, eq, eq_count, balance
    )

def _safe_float(value, default):
    return float(default) if value is None else float(value)

def run_backtest(timeframe: str, df: pd.DataFrame, signals: pd.Series, entry_params: dict, exit_model: str, exit_params: dict):
    exit_model_map = {
        "fixed_tp": 0, "mr_exit": 1, "fixed_tp_plus_mr": 2, 
        "partial_tp_plus_mr": 3, "partial_tp_mr_time_stop": 4
    }
    exit_mode_code = int(exit_model_map[exit_model])
    leg_a_atr_target = _safe_float(exit_params.get("leg_a_atr_target", -1.0), -1.0) if exit_params else -1.0
    time_stop_minutes = _safe_float(exit_params.get("time_stop_minutes", -1.0), -1.0) if exit_params else -1.0
    trail_mult = _safe_float(exit_params.get("trail_mult", 0.0), 0.0) if exit_params else 0.0

    idx = df.index
    time_minutes = (idx.view("int64") // 60000000000).astype(np.int64)

    sig = signals.reindex(idx).fillna(0).astype(np.int8).to_numpy()
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    close = df["close"].to_numpy(dtype=np.float64)
    spread = df["spread"].fillna(SPREAD_CAP_POINTS).to_numpy(dtype=np.float64)
    atr14 = df["atr14"].to_numpy(dtype=np.float64)
    regime_code = df["regime_code"].fillna(3).to_numpy(dtype=np.int32)
    
    is_m5 = 1 if timeframe == "M5" else 0

    (
        trade_entry_idx, trade_exit_idx, trade_leg, trade_side, trade_entry_px,
        trade_exit_px, trade_move_pips, trade_pnl_cents, trade_reason,
        trade_entry_regime, trade_count, eq, eq_count, ending_balance
    ) = _run_backtest_numba(
        sig, high, low, close, spread, atr14, regime_code, time_minutes,
        float(entry_params["atr_stop"]), float(entry_params["atr_target"]),
        leg_a_atr_target, exit_mode_code, time_stop_minutes, trail_mult,
        float(INITIAL_BALANCE_CENTS), float(PIP_SIZE_PRICE), float(PIP_VALUE_CENTS_PER_1LOT),
        float(SLIPPAGE_PIPS), float(SPREAD_CAP_POINTS), float(COMMISSION_CENTS_PER_TRADE),
        int(is_m5)
    )

    if trade_count == 0:
        return pd.DataFrame(), compute_metrics(pd.DataFrame(), [], INITIAL_BALANCE_CENTS, float(ending_balance))

    reasons = {1: "STOP", 2: "PROFIT_TARGET", 3: "FIXED_TP", 4: "MR_EXIT", 5: "TIME_STOP", 6: "EOD_CLOSE", 7: "REGIME_SHIFT"}
    regime_map = {1: "TREND", 2: "SHOCK", 3: "MR"}

    trades_df = pd.DataFrame(
        {
            "entry_time": idx[trade_entry_idx[:trade_count]],
            "exit_time": idx[trade_exit_idx[:trade_count]],
            "leg": np.where(trade_leg[:trade_count] == 0, "A", np.where(trade_leg[:trade_count] == 1, "B", "C_SCALE")),
            "side": np.where(trade_side[:trade_count] == 1, "BUY", "SELL"),
            "entry_price": trade_entry_px[:trade_count],
            "exit_price": trade_exit_px[:trade_count],
            "move_pips": trade_move_pips[:trade_count],
            "pnl_cents": trade_pnl_cents[:trade_count],
            "exit_reason": [reasons[int(x)] for x in trade_reason[:trade_count]],
            "entry_regime": [regime_map[int(x)] for x in trade_entry_regime[:trade_count]],
            "exit_model": exit_model,
        }
    )

    return trades_df, compute_metrics(trades_df, eq[:eq_count].tolist(), INITIAL_BALANCE_CENTS, float(ending_balance))

In [7]:
# Vectorized Plateau Stability Engine & Group Experiment Runner

import zlib

def _is_numeric_grid(values: list) -> bool:
    return all(isinstance(v, (int, float, np.integer, np.floating)) for v in values)

def build_step_map(param_grid: dict) -> dict:
    step_map = {}
    for key, vals in param_grid.items():
        uniq = sorted(set(vals), key=lambda x: str(x))
        if _is_numeric_grid(uniq) and len(uniq) > 1:
            diffs = np.diff(np.array(uniq, dtype=float))
            diffs = diffs[diffs > 0]
            step_map[key] = float(np.min(diffs)) if len(diffs) > 0 else 1.0
        else:
            step_map[key] = 1.0 if _is_numeric_grid(uniq) else None
    return step_map

def add_parameter_stability_score(
    results_df: pd.DataFrame,
    param_cols: list[str],
    step_map: dict,
    perf_col: str = "profit_per_trade",
    block_size: int = 128,  # lower to reduce memory pressure
) -> pd.DataFrame:
    if results_df.empty:
        out = results_df.copy()
        out["parameter_stability_score"] = np.nan
        return out

    out = results_df.copy()
    M = len(out)

    numeric_cols = [c for c in param_cols if step_map.get(c, None) is not None]
    cat_cols = [c for c in param_cols if step_map.get(c, None) is None]

    num_mat = out[numeric_cols].astype(float).to_numpy() if numeric_cols else np.empty((M, 0), dtype=float)
    step_sizes = np.array([float(step_map[c]) for c in numeric_cols], dtype=float) if numeric_cols else np.empty((0,), dtype=float)
    cat_mat = np.column_stack([pd.factorize(out[c].astype(str))[0] for c in cat_cols]).astype(np.int32) if cat_cols else np.empty((M, 0), dtype=np.int32)

    perf = out[perf_col].astype(float).to_numpy()
    perf2 = perf * perf
    scores = np.empty(M, dtype=float)

    for start in range(0, M, block_size):
        end = min(M, start + block_size)

        if num_mat.shape[1] > 0:
            diffs = np.abs(num_mat[start:end, None, :] - num_mat[None, :, :])
            num_mask = np.all(diffs <= step_sizes, axis=2)
        else:
            num_mask = np.ones((end - start, M), dtype=bool)

        if cat_mat.shape[1] > 0:
            cat_mask = np.all(cat_mat[start:end, None, :] == cat_mat[None, :, :], axis=2)
        else:
            cat_mask = np.ones((end - start, M), dtype=bool)

        mask = num_mask & cat_mask
        counts = mask.sum(axis=1).astype(np.float64)
        sums = mask @ perf
        sums2 = mask @ perf2

        means = np.where(counts > 0, sums / counts, -1e9)
        vars_ = np.maximum(np.where(counts > 0, sums2 / counts - means * means, 0.0), 0.0)
        scores[start:end] = means - np.sqrt(vars_)

    out["parameter_stability_score"] = scores
    return out

def _cap_entry_combos(
    entry_combos: list[dict],
    timeframe: str,
    strategy_name: str,
    exit_model: str,
) -> tuple[list[dict], int, bool]:
    enable = bool(globals().get("ENABLE_ENTRY_CAP", False))
    cap_map = globals().get("ENTRY_CAP_BY_TF", {})
    cap_val = cap_map.get(timeframe, None) if isinstance(cap_map, dict) else None

    original_n = len(entry_combos)
    if (not enable) or (cap_val is None):
        return entry_combos, original_n, False

    cap_n = int(cap_val)
    if cap_n <= 0 or original_n <= cap_n:
        return entry_combos, original_n, False

    seed_base = int(globals().get("ENTRY_CAP_SEED_BASE", 42))
    seed_key = f"{seed_base}|{timeframe}|{strategy_name}|{exit_model}"
    seed = zlib.crc32(seed_key.encode("utf-8")) & 0xFFFFFFFF

    rng = np.random.default_rng(seed)
    picked_idx = np.sort(rng.choice(original_n, size=cap_n, replace=False))
    capped = [entry_combos[i] for i in picked_idx.tolist()]
    return capped, original_n, True

def get_exit_grid_for_mode(timeframe: str, exit_model: str):
    ts_grid = TIME_STOP_GRID_BY_TF[timeframe]
    tr_grid = TRAIL_MULT_GRID

    if exit_model == "fixed_tp":
        cfg = [{"leg_a_atr_target": p} for p in LEG_A_ATR_TARGET_GRID]
        grid_map = {"leg_a_atr_target": LEG_A_ATR_TARGET_GRID}
    elif exit_model == "mr_exit":
        cfg = [{"leg_a_atr_target": None}]
        grid_map = {}
    elif exit_model == "fixed_tp_plus_mr":
        cfg = [{"leg_a_atr_target": p} for p in LEG_A_ATR_TARGET_GRID]
        grid_map = {"leg_a_atr_target": LEG_A_ATR_TARGET_GRID}
    elif exit_model == "partial_tp_plus_mr":
        cfg = [{"leg_a_atr_target": p} for p in LEG_A_ATR_TARGET_GRID]
        grid_map = {"leg_a_atr_target": LEG_A_ATR_TARGET_GRID}
    elif exit_model == "partial_tp_mr_time_stop":
        cfg = [
            {
                "leg_a_atr_target": p,
                "time_stop_minutes": t,
                "trail_mult": m,
            }
            for p, t, m in itertools.product(
                LEG_A_ATR_TARGET_GRID,
                ts_grid,
                tr_grid,
            )
        ]
        grid_map = {
            "leg_a_atr_target": LEG_A_ATR_TARGET_GRID,
            "time_stop_minutes": ts_grid,
            "trail_mult": tr_grid,
        }
    else:
        raise ValueError(f"Unsupported exit model: {exit_model}")

    return cfg, grid_map

def run_group(timeframe: str, strategy_name: str, exit_model: str) -> pd.DataFrame:
    df = FEATURES_BY_TF[timeframe]
    strategy = STRATEGIES[strategy_name]

    # Per-TF grid override: rebind the per-strategy param_grid and module-level exit grids
    # for M5 only, so M15 runs are byte-identical to before this change.
    _restore = False
    _orig_grid = dict(strategy.param_grid)
    _orig_legA = list(LEG_A_ATR_TARGET_GRID)
    _orig_entry = list(ENTRY_ATR_TARGET_GRID)
    _orig_atrT = list(ATR_TARGET_GRID)
    if timeframe == "M5":
        if strategy_name == "trend_pullback":
            strategy.param_grid = {**_orig_grid,
                "adx_threshold":    M5_ADX_GRID,
                "pullback_rsi":     M5_PULLBACK_RSI_GRID,
                "confirmation_bars":M5_CONFIRMATION_GRID,
                "atr_stop":         M5_ATR_STOP_GRID,
                "atr_target":       M5_ENTRY_TARGET_GRID,
                "session_filter":   _orig_grid.get("session_filter", SESSION_FILTER_VALUES),
            }
        elif strategy_name == "volatility_expansion":
            strategy.param_grid = {**_orig_grid,
                "atr_stop":         M5_ATR_STOP_GRID,
                "atr_target":       M5_ENTRY_TARGET_GRID,
                "session_filter":   _orig_grid.get("session_filter", SESSION_FILTER_VALUES),
            }
        LEG_A_ATR_TARGET_GRID[:] = M5_LEG_A_TARGET_GRID
        ENTRY_ATR_TARGET_GRID[:] = M5_ENTRY_TARGET_GRID
        ATR_TARGET_GRID[:]       = M5_LEG_A_TARGET_GRID
        _restore = True

    t0 = time.time()
    try:
        entry_combos_full = list(strategy.iter_param_dicts())
        entry_combos, full_n, was_capped = _cap_entry_combos(
            entry_combos_full, timeframe, strategy_name, exit_model
        )

        exit_cfgs, exit_grid_map = get_exit_grid_for_mode(timeframe, exit_model)

        rows = []
        total = len(entry_combos) * len(exit_cfgs)
        done = 0

        if was_capped:
            print(
                f"[{timeframe}][{strategy_name}][{exit_model}] entry combos capped: "
                f"{len(entry_combos)}/{full_n}"
            )
        else:
            print(
                f"[{timeframe}][{strategy_name}][{exit_model}] entry combos used: "
                f"{len(entry_combos)}/{full_n}"
            )

        for i, entry_params in enumerate(entry_combos, 1):
            signals = generate_routed_signals(df, entry_params, strategy_name)

            for exit_cfg in exit_cfgs:
                _, metrics = run_backtest(
                    timeframe=timeframe,
                    df=df,
                    signals=signals,
                    entry_params=entry_params,
                    exit_model=exit_model,
                    exit_params=exit_cfg,
                )

                combined_params = {
                    **entry_params,
                    **{k: v for k, v in exit_cfg.items() if v is not None},
                }

                row = {
                    "timeframe": timeframe,
                    "strategy_name": strategy_name,
                    "exit_model": exit_model,
                    "parameter_set": json.dumps(combined_params, sort_keys=True),
                    **metrics,
                }
                for k, v in combined_params.items():
                    row[k] = v
                rows.append(row)

                done += 1

            if i % 50 == 0 or i == len(entry_combos):
                print(f"[{timeframe}][{strategy_name}][{exit_model}] {done}/{total}")

        res = pd.DataFrame(rows)

        param_cols = list(strategy.param_cols) + list(exit_grid_map.keys())
        full_grid_map = {**strategy.param_grid, **exit_grid_map}
        step_map = build_step_map(full_grid_map)

        res = add_parameter_stability_score(
            res, param_cols=param_cols, step_map=step_map, perf_col="profit_per_trade"
        )

        res["robust_score"] = (
            0.45 * res["parameter_stability_score"].astype(float)
            + 0.35 * res["profit_per_trade"].astype(float)
            + 0.20 * res["profit_factor"].astype(float)
        )

        res = res.sort_values(
            ["robust_score", "profit_per_trade", "profit_factor", "sharpe"],
            ascending=False,
        ).reset_index(drop=True)

        return res
    finally:
        if _restore:
            strategy.param_grid = _orig_grid
            LEG_A_ATR_TARGET_GRID[:] = _orig_legA
            ENTRY_ATR_TARGET_GRID[:] = _orig_entry
            ATR_TARGET_GRID[:]       = _orig_atrT
        elapsed = time.time() - t0
        print(f"[{timeframe}][{strategy_name}][{exit_model}] done in {elapsed/60:.2f} minutes")


In [8]:
# Concurrent execution across timeframe + strategy + exit model

group_tasks = [
    (tf, strategy_name, exit_model)
    for tf in TIMEFRAMES
    for strategy_name in STRATEGIES.keys()
    for exit_model in EXIT_MODELS
]

print("Total groups:", len(group_tasks))
for t in group_tasks:
    print("Task:", t)

def _run_task(task):
    tf, strategy_name, exit_model = task
    return run_group(tf, strategy_name, exit_model)

if JOBLIB_OK:
    try:
        nj = min(max(1, N_JOBS), len(group_tasks))
        print(f"Running parallel with joblib loky backend... n_jobs={nj}")
        group_results = Parallel(n_jobs=nj, backend="loky", verbose=10, batch_size=1)(
            delayed(_run_task)(task) for task in group_tasks
        )
    except Exception as e:
        print(f"[warn] loky failed: {type(e).__name__}: {e}")
        try:
            nj = min(max(1, N_JOBS), len(group_tasks))
            print(f"Falling back to threading... n_jobs={nj}")
            group_results = Parallel(n_jobs=nj, backend="threading", verbose=10, batch_size=1)(
                delayed(_run_task)(task) for task in group_tasks
            )
        except Exception as e2:
            print(f"[warn] threading failed: {type(e2).__name__}: {e2}")
            print("Falling back to sequential execution.")
            group_results = [_run_task(task) for task in group_tasks]
else:
    print("joblib unavailable. Running sequentially.")
    group_results = [_run_task(task) for task in group_tasks]

all_results = pd.concat(group_results, axis=0, ignore_index=True)

# Apply statistical significance filter before sorting
valid_results = all_results[all_results["trade_count"] >= 200]

best_rows = (
    valid_results.sort_values(
        ["robust_score", "profit_per_trade", "profit_factor", "sharpe"],
        ascending=False,
    )
    .groupby(["timeframe", "strategy_name", "exit_model"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print("All results shape:", all_results.shape)
print("Best rows shape:", best_rows.shape)
display(best_rows.head(20))

Total groups: 20
Task: ('M15', 'trend_pullback', 'fixed_tp')
Task: ('M15', 'trend_pullback', 'mr_exit')
Task: ('M15', 'trend_pullback', 'fixed_tp_plus_mr')
Task: ('M15', 'trend_pullback', 'partial_tp_plus_mr')
Task: ('M15', 'trend_pullback', 'partial_tp_mr_time_stop')
Task: ('M15', 'volatility_expansion', 'fixed_tp')
Task: ('M15', 'volatility_expansion', 'mr_exit')
Task: ('M15', 'volatility_expansion', 'fixed_tp_plus_mr')
Task: ('M15', 'volatility_expansion', 'partial_tp_plus_mr')
Task: ('M15', 'volatility_expansion', 'partial_tp_mr_time_stop')
Task: ('M5', 'trend_pullback', 'fixed_tp')
Task: ('M5', 'trend_pullback', 'mr_exit')
Task: ('M5', 'trend_pullback', 'fixed_tp_plus_mr')
Task: ('M5', 'trend_pullback', 'partial_tp_plus_mr')
Task: ('M5', 'trend_pullback', 'partial_tp_mr_time_stop')
Task: ('M5', 'volatility_expansion', 'fixed_tp')
Task: ('M5', 'volatility_expansion', 'mr_exit')
Task: ('M5', 'volatility_expansion', 'fixed_tp_plus_mr')
Task: ('M5', 'volatility_expansion', 'partial_tp

[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed:  1.7min
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:  6.3min
[Parallel(n_jobs=6)]: Done  12 out of  20 | elapsed:  8.3min remaining:  5.6min
[Parallel(n_jobs=6)]: Done  15 out of  20 | elapsed:  9.7min remaining:  3.2min
[Parallel(n_jobs=6)]: Done  18 out of  20 | elapsed: 16.0min remaining:  1.8min
[Parallel(n_jobs=6)]: Done  20 out of  20 | elapsed: 40.2min finished


All results shape: (180512, 29)
Best rows shape: (20, 29)


,timeframe,strategy_name,exit_model,parameter_set,profit_factor,sharpe,sortino,calmar,max_drawdown,expectancy,...,atr_target,session_filter,leg_a_atr_target,parameter_stability_score,robust_score,time_stop_minutes,trail_mult,atr_expansion_threshold,breakout_lookback,breakout_buffer
0,M15,trend_pullback,fixed_tp,"{""adx_threshold"": 18.0, ""atr_stop"": 3.0, ""atr_...",1.474333,6.328621,9.919439,77.589480,25.005495,13.580238,...,2.0,NY,1.5,7.593952,8.465228,NaN,NaN,NaN,NaN,NaN
1,M15,trend_pullback,fixed_tp_plus_mr,"{""adx_threshold"": 15.0, ""atr_stop"": 3.0, ""atr_...",1.565720,7.864580,10.041143,127.161641,16.923519,11.541057,...,2.0,NY,1.0,7.359013,7.664070,NaN,NaN,NaN,NaN,NaN
2,M15,trend_pullback,mr_exit,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",1.242302,1.482546,6.347835,5.749435,195.932006,12.743193,...,2.5,NY,NaN,4.473938,6.721850,NaN,NaN,NaN,NaN,NaN
3,M15,trend_pullback,partial_tp_plus_mr,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",1.297962,1.867466,5.970224,14.250701,74.579759,8.956297,...,2.0,NY,1.0,3.862602,5.132467,NaN,NaN,NaN,NaN,NaN
4,M5,trend_pullback,fixed_tp,"{""adx_threshold"": 20, ""atr_stop"": 3.0, ""atr_ta...",1.578148,12.286608,10.897269,136.341289,23.467935,7.441043,...,1.0,NY,1.0,4.397669,4.898946,NaN,NaN,NaN,NaN,NaN
5,M5,trend_pullback,fixed_tp_plus_mr,"{""adx_threshold"": 25, ""atr_stop"": 3.0, ""atr_ta...",1.691517,11.900985,10.029855,112.408222,21.114864,7.245068,...,1.0,NY,0.8,4.382946,4.846403,NaN,NaN,NaN,NaN,NaN
6,M15,trend_pullback,partial_tp_mr_time_stop,"{""adx_threshold"": 20.0, ""atr_stop"": 3.0, ""atr_...",1.492644,0.879035,20.063645,5.757605,207.567368,8.774529,...,2.0,London_NY,1.5,-1.152407,2.851031,120.0,2.5,NaN,NaN,NaN
7,M5,trend_pullback,partial_tp_mr_time_stop,"{""adx_threshold"": 20, ""atr_stop"": 1.5, ""atr_ta...",1.948000,0.733145,45.608003,3.176689,371.957248,8.897535,...,1.0,NY,0.8,-1.592991,2.786891,60.0,2.5,NaN,NaN,NaN
8,M5,trend_pullback,mr_exit,"{""adx_threshold"": 20, ""atr_stop"": 3.0, ""atr_ta...",1.135441,1.293377,6.368363,4.282328,265.733850,5.436113,...,1.0,NY,NaN,1.293878,2.711973,NaN,NaN,NaN,NaN,NaN
9,M5,trend_pullback,partial_tp_plus_mr,"{""adx_threshold"": 25, ""atr_stop"": 2.0, ""atr_ta...",1.137192,1.110382,6.097144,3.320907,208.158148,2.035553,...,1.0,London,0.8,-0.824957,0.568651,NaN,NaN,NaN,NaN,NaN


In [9]:
# Reporting and leaderboards (use valid_results to exclude low-trade runs)

os.makedirs("reports", exist_ok=True)
leg_c_lot_rule = "A_if_A_hits_first_else_B"
# Required strategy summary (plus useful extras) -- filtered
summary_cols = [
    "timeframe",
    "strategy_name",
    "exit_model",
    "profit_factor",
    "sharpe",
    "sortino",
    "calmar",
    "max_drawdown",
    "expectancy",
    "win_rate",
    "trade_count",
    "parameter_set",
    "profit_per_trade",
    "parameter_stability_score",
    "robust_score",
]
strategy_summary = valid_results[summary_cols].copy()
strategy_summary["leg_c_lot_rule"] = leg_c_lot_rule
strategy_summary.to_csv("reports/strategy_summary.csv", index=False)

# Full output for diagnostics (unfiltered)
all_results.to_csv("reports/strategy_full_results.csv", index=False)

# New required leaderboard format -- filtered
leaderboard = (
    valid_results.sort_values(
        ["profit_per_trade", "robust_score", "profit_factor", "sharpe"],
        ascending=False,
    )
    .copy()
)
leaderboard = leaderboard.assign(leg_c_lot_rule=leg_c_lot_rule)
leaderboard_view = leaderboard[
    [
        "timeframe",
        "strategy_name",
        "exit_model",
        "profit_factor",
        "sharpe",
        "sortino",
        "calmar",
        "expectancy",
        "profit_per_trade",
        "trade_count",
        "max_drawdown",
        "parameter_set",
        "parameter_stability_score",
        "robust_score",
    ]
].copy()

leaderboard_view = leaderboard_view.rename(
    columns={
        "timeframe": "Timeframe",
        "strategy_name": "Strategy",
        "exit_model": "Exit Model",
        "profit_factor": "PF",
        "sharpe": "Sharpe",
        "sortino": "Sortino",
        "calmar": "Calmar",
        "expectancy": "Expectancy",
        "profit_per_trade": "Profit Per Trade",
        "trade_count": "Trade Count",
        "max_drawdown": "Max DD",
        "leg_c_lot_rule": "Leg C Lot Rule",
    },
)

leaderboard_view.to_csv("reports/strategy_leaderboard.csv", index=False)

# Focus comparison requested in objective -- filtered
focus = valid_results[
    (valid_results["strategy_name"].isin(["trend_pullback", "volatility_expansion"]))
    & (
        valid_results["exit_model"].isin(
            ["partial_tp_plus_mr", "partial_tp_mr_time_stop"]
        )
    )
].copy()

focus = focus.sort_values(
    ["timeframe", "profit_per_trade", "robust_score", "profit_factor", "sharpe"],
    ascending=[True, False, False, False, False],
).reset_index(drop=True)
focus["leg_c_lot_rule"] = leg_c_lot_rule
focus.to_csv("reports/strategy_focus_partialtp_mr.csv", index=False)

print("Saved: reports/strategy_summary.csv")
print("Saved: reports/strategy_full_results.csv")
print("Saved: reports/strategy_leaderboard.csv")
print("Saved: reports/strategy_focus_partialtp_mr.csv")

print("\nTop Leaderboard Rows")
display(leaderboard_view.head(30))

print("\nTop Focus Rows (partial TP + MR variants)")
display(
    focus[
        [
            "timeframe",
            "strategy_name",
            "exit_model",
            "profit_factor",
            "sharpe",
            "sortino",
            "calmar",
            "expectancy",
            "profit_per_trade",
            "trade_count",
            "max_drawdown",
            "parameter_set",
            "parameter_stability_score",
            "robust_score",
            "leg_c_lot_rule",
        ]
    ].head(30)
)

Saved: reports/strategy_summary.csv
Saved: reports/strategy_full_results.csv
Saved: reports/strategy_leaderboard.csv
Saved: reports/strategy_focus_partialtp_mr.csv

Top Leaderboard Rows


,Timeframe,Strategy,Exit Model,PF,Sharpe,Sortino,Calmar,Expectancy,Profit Per Trade,Trade Count,Max DD,parameter_set,parameter_stability_score,robust_score
0,M15,trend_pullback,fixed_tp,1.474333,6.328621,9.919439,77.589480,13.580238,13.580238,2143,25.005495,"{""adx_threshold"": 18.0, ""atr_stop"": 3.0, ""atr_...",7.593952,8.465228
3,M15,trend_pullback,fixed_tp,1.455073,6.521099,10.217821,90.635954,13.123564,13.123564,2396,23.128466,"{""adx_threshold"": 15.0, ""atr_stop"": 3.0, ""atr_...",7.447712,8.235732
5,M15,trend_pullback,fixed_tp,1.442676,6.271527,9.831851,85.845945,12.830532,12.830532,2328,23.196186,"{""adx_threshold"": 15.0, ""atr_stop"": 3.0, ""atr_...",7.447712,8.130692
8640,M15,trend_pullback,mr_exit,1.242302,1.482546,6.347835,5.749435,12.743193,12.743193,1326,195.932006,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",4.473938,6.721850
8641,M15,trend_pullback,mr_exit,1.242302,1.482546,6.347835,5.749435,12.743193,12.743193,1326,195.932006,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",4.473938,6.721850
8642,M15,trend_pullback,mr_exit,1.242302,1.482546,6.347835,5.749435,12.743193,12.743193,1326,195.932006,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",4.473938,6.721850
1,M15,trend_pullback,fixed_tp,1.566285,7.816743,10.448553,146.011616,12.303197,12.303197,2778,15.605280,"{""adx_threshold"": 15.0, ""atr_stop"": 3.0, ""atr_...",8.411889,8.404726
11523,M15,trend_pullback,fixed_tp_plus_mr,1.444222,6.037194,8.799467,64.292577,12.163835,12.163835,2152,27.143177,"{""adx_threshold"": 18.0, ""atr_stop"": 3.0, ""atr_...",6.027943,7.258761
8643,M15,trend_pullback,mr_exit,1.225196,1.345072,5.605272,6.144600,12.091151,12.091151,1252,164.243101,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",4.473938,6.490214
8644,M15,trend_pullback,mr_exit,1.225196,1.345072,5.605272,6.144600,12.091151,12.091151,1252,164.243101,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",4.473938,6.490214



Top Focus Rows (partial TP + MR variants)


,timeframe,strategy_name,exit_model,profit_factor,sharpe,sortino,calmar,expectancy,profit_per_trade,trade_count,max_drawdown,parameter_set,parameter_stability_score,robust_score,leg_c_lot_rule
0,M15,trend_pullback,partial_tp_mr_time_stop,1.512965,0.784415,20.083765,3.684594,9.200445,9.200445,1737,289.153005,"{""adx_threshold"": 20.0, ""atr_stop"": 3.0, ""atr_...",-1.859744,2.685864,A_if_A_hits_first_else_B
1,M15,trend_pullback,partial_tp_mr_time_stop,1.512965,0.784415,20.083765,3.684594,9.200445,9.200445,1737,289.153005,"{""adx_threshold"": 20.0, ""atr_stop"": 3.0, ""atr_...",-1.859744,2.685864,A_if_A_hits_first_else_B
2,M15,trend_pullback,partial_tp_mr_time_stop,1.512965,0.784415,20.083765,3.684594,9.200445,9.200445,1737,289.153005,"{""adx_threshold"": 20.0, ""atr_stop"": 3.0, ""atr_...",-1.859744,2.685864,A_if_A_hits_first_else_B
3,M15,trend_pullback,partial_tp_mr_time_stop,1.512965,0.784415,20.083765,3.684594,9.200445,9.200445,1737,289.153005,"{""adx_threshold"": 20.0, ""atr_stop"": 3.0, ""atr_...",-1.859744,2.685864,A_if_A_hits_first_else_B
4,M15,trend_pullback,partial_tp_mr_time_stop,1.512965,0.784415,20.083765,3.684594,9.200445,9.200445,1737,289.153005,"{""adx_threshold"": 20.0, ""atr_stop"": 3.0, ""atr_...",-1.859744,2.685864,A_if_A_hits_first_else_B
5,M15,trend_pullback,partial_tp_mr_time_stop,1.512965,0.784415,20.083765,3.684594,9.200445,9.200445,1737,289.153005,"{""adx_threshold"": 20.0, ""atr_stop"": 3.0, ""atr_...",-1.859744,2.685864,A_if_A_hits_first_else_B
6,M15,trend_pullback,partial_tp_mr_time_stop,1.512965,0.784415,20.083765,3.684594,9.200445,9.200445,1737,289.153005,"{""adx_threshold"": 20.0, ""atr_stop"": 3.0, ""atr_...",-1.859744,2.685864,A_if_A_hits_first_else_B
7,M15,trend_pullback,partial_tp_mr_time_stop,1.512965,0.784415,20.083765,3.684594,9.200445,9.200445,1737,289.153005,"{""adx_threshold"": 20.0, ""atr_stop"": 3.0, ""atr_...",-1.859744,2.685864,A_if_A_hits_first_else_B
8,M15,trend_pullback,partial_tp_mr_time_stop,1.512965,0.784415,20.083765,3.684594,9.200445,9.200445,1737,289.153005,"{""adx_threshold"": 20.0, ""atr_stop"": 3.0, ""atr_...",-1.859744,2.685864,A_if_A_hits_first_else_B
9,M15,trend_pullback,partial_tp_mr_time_stop,1.449146,0.737094,20.102038,2.851875,9.194534,9.194534,1634,351.204475,"{""adx_threshold"": 20.0, ""atr_stop"": 3.0, ""atr_...",-2.771094,2.260924,A_if_A_hits_first_else_B


In [10]:
# Export top robust candidates for downstream wiring into GoldRegimeX_Explorer
# FIX (2026-07-07): previously exported only the top 30 rows by profit_per_trade,
# which meant strategies that survived the 25% Max DD filter but had lower
# profit-per-trade were dropped from the CSV, causing the Explorer to raise
# "No baseline survived 25%% DD filter" even when survivors clearly existed
# (see Cell 12 output). Now the CSV always includes every DD-surviving row
# from valid_results, plus the top N by profit_per_trade for downstream
# ranking flexibility.

leg_c_lot_rule = "A_if_A_hits_first_else_B"
HANDOFF_MAX_DD_PCT = 25.0

handoff_cols = [
    "timeframe",
    "strategy_name",
    "exit_model",
    "parameter_set",
    "profit_factor",
    "sharpe",
    "sortino",
    "calmar",
    "max_drawdown",
    "expectancy",
    "win_rate",
    "trade_count",
    "profit_per_trade",
    "parameter_stability_score",
    "robust_score",
    "leg_c_lot_rule",
]

top_n = 30

# 1) top-N by profit_per_trade / robust_score (previous behaviour)
top_by_profit = (
    valid_results.sort_values(
        ["profit_per_trade", "robust_score", "profit_factor", "sharpe"],
        ascending=False,
    )
    .head(top_n)
    .copy()
)

# 2) ALL rows that survive the 25% Max DD cap - the Explorer needs these.
dd_survivors = valid_results[valid_results["max_drawdown"] <= HANDOFF_MAX_DD_PCT].copy()

# 3) For each timeframe, guarantee at least the top-K DD-survivors by robust_score
#    are present, so both M15 and M5 have baselines to hand off.
per_tf_k = 10
per_tf_survivors = (
    dd_survivors
    .sort_values(["robust_score", "profit_per_trade"], ascending=False)
    .groupby("timeframe", as_index=False)
    .head(per_tf_k)
)

# 4) Union (DD-survivors first so they sort earliest by construction), then dedupe.
handoff = pd.concat([per_tf_survivors, dd_survivors, top_by_profit], ignore_index=True)
handoff = handoff.drop_duplicates(
    subset=["timeframe", "strategy_name", "exit_model", "parameter_set"],
    keep="first",
).reset_index(drop=True)
handoff["leg_c_lot_rule"] = leg_c_lot_rule
handoff = handoff[handoff_cols].copy()

import os
os.makedirs("reports", exist_ok=True)
handoff.to_csv("reports/strategy_winners_for_explorer.csv", index=False)
print("Saved: reports/strategy_winners_for_explorer.csv")
print("  Total rows:", len(handoff))
print("  DD<=%.1f%% rows:" % HANDOFF_MAX_DD_PCT,
      int((handoff["max_drawdown"] <= HANDOFF_MAX_DD_PCT).sum()))
print("  Per timeframe DD survivors:")
print(handoff[handoff["max_drawdown"] <= HANDOFF_MAX_DD_PCT]
      .groupby("timeframe").size().to_string())
display(handoff)


Saved: reports/strategy_winners_for_explorer.csv
  Total rows: 757
  DD<=25.0% rows: 737
  Per timeframe DD survivors:
timeframe
M15    326
M5     411


,timeframe,strategy_name,exit_model,parameter_set,profit_factor,sharpe,sortino,calmar,max_drawdown,expectancy,win_rate,trade_count,profit_per_trade,parameter_stability_score,robust_score,leg_c_lot_rule
0,M15,trend_pullback,fixed_tp,"{""adx_threshold"": 15.0, ""atr_stop"": 3.0, ""atr_...",1.566285,7.816743,10.448553,146.011616,15.605280,12.303197,0.628870,2778,12.303197,8.411889,8.404726,A_if_A_hits_first_else_B
1,M15,trend_pullback,fixed_tp,"{""adx_threshold"": 18.0, ""atr_stop"": 3.0, ""atr_...",1.547941,7.130244,9.490038,120.263943,16.543264,12.004575,0.629525,2486,12.004575,8.396518,8.289623,A_if_A_hits_first_else_B
2,M15,trend_pullback,fixed_tp,"{""adx_threshold"": 15.0, ""atr_stop"": 3.0, ""atr_...",1.455073,6.521099,10.217821,90.635954,23.128466,13.123564,0.591820,2396,13.123564,7.447712,8.235732,A_if_A_hits_first_else_B
3,M15,trend_pullback,fixed_tp,"{""adx_threshold"": 15.0, ""atr_stop"": 3.0, ""atr_...",1.533512,7.316974,9.749876,134.606932,15.687473,11.766210,0.627043,2692,11.766210,8.411889,8.210226,A_if_A_hits_first_else_B
4,M15,trend_pullback,fixed_tp,"{""adx_threshold"": 15.0, ""atr_stop"": 3.0, ""atr_...",1.442676,6.271527,9.831851,85.845945,23.196186,12.830532,0.591924,2328,12.830532,7.447712,8.130692,A_if_A_hits_first_else_B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
752,M15,trend_pullback,fixed_tp,"{""adx_threshold"": 20.0, ""atr_stop"": 3.0, ""atr_...",1.360974,4.728240,6.980239,47.031191,28.677438,10.784121,0.593817,1876,10.784121,7.593952,7.463916,A_if_A_hits_first_else_B
753,M15,trend_pullback,fixed_tp,"{""adx_threshold"": 18.0, ""atr_stop"": 3.0, ""atr_...",1.300130,4.025204,6.671797,40.414353,32.646039,10.686030,0.553996,1852,10.686030,7.345820,7.305756,A_if_A_hits_first_else_B
754,M15,trend_pullback,fixed_tp,"{""adx_threshold"": 20.0, ""atr_stop"": 3.0, ""atr_...",1.350328,5.294577,8.056982,43.969208,41.844983,10.635207,0.576493,2595,10.635207,5.476443,6.456787,A_if_A_hits_first_else_B
755,M15,trend_pullback,fixed_tp,"{""adx_threshold"": 18.0, ""atr_stop"": 3.0, ""atr_...",1.342942,5.379624,8.061117,47.892417,41.341375,10.614395,0.580057,2798,10.614395,5.476443,6.448026,A_if_A_hits_first_else_B


In [11]:
# Regime Performance Attribution Analytics Report

def generate_regime_report(trades_df: pd.DataFrame):
    if trades_df.empty:
        print("Regime Report Error: Zero transaction logs generated.")
        return

    regimes = ["TREND", "SHOCK", "MR"]
    print("=" * 50)
    print("      REGIME RESEARCH ATTRIBUTION SUMMARY      ")
    print("=" * 50)

    total_pnl = trades_df["pnl_cents"].sum()

    for regime in regimes:
        r_trades = trades_df[trades_df["entry_regime"] == regime]

        if r_trades.empty:
            print(f"\nREGIME: {regime}\n" + "-" * 20)
            print("Buys: 0 | Sells: 0\nNet Profit: 0.00 Cents\nProfit Factor: N/A\nPnL Contribution: 0.0%")
            continue

        buys = len(r_trades[r_trades["side"] == "BUY"])
        sells = len(r_trades[r_trades["side"] == "SELL"])

        r_pnl = r_trades["pnl_cents"].to_numpy()
        gross_profit = np.sum(r_pnl[r_pnl > 0])
        gross_loss = -np.sum(r_pnl[r_pnl < 0])

        pf = gross_profit / gross_loss if gross_loss > 0 else (10.0 if gross_profit > 0 else 0.0)
        net_profit = np.sum(r_pnl)
        contribution = (net_profit / total_pnl * 100.0) if total_pnl != 0 else 0.0

        print(f"\nREGIME: {regime}\n" + "-" * 20)
        print(f"Buys: {buys} | Sells: {sells}")
        print(f"Net Profit: {net_profit:.2f} Cents")
        print(f"Profit Factor: {pf:.2f}")
        print(f"PnL Contribution: {contribution:.1f}%")

    print("=" * 50)


# Run a regime report on the top valid run for EACH Timeframe and EACH Strategy
if valid_results.empty:
    print("No valid_results available (trade_count >= 200). Run the backtest cell first.")
else:
    for tf in TIMEFRAMES:
        for strat_name in STRATEGIES.keys():
            # Filter for specific timeframe and strategy
            strat_results = valid_results[
                (valid_results["strategy_name"] == strat_name)
                & (valid_results["timeframe"] == tf)
            ]

            if strat_results.empty:
                print(f"\nNo valid runs found for {tf} | {strat_name}.")
                continue

            best_strat_row = strat_results.sort_values(
                ["robust_score", "profit_per_trade", "profit_factor", "sharpe"],
                ascending=False,
            ).iloc[0]

            exit_model = best_strat_row["exit_model"]

            combined_params = json.loads(best_strat_row["parameter_set"])
            entry_params = {
                k: combined_params[k]
                for k in STRATEGIES[strat_name].param_cols
                if k in combined_params
            }

            exit_keys = ["leg_a_atr_target", "time_stop_minutes", "trail_mult"]
            exit_params = {k: combined_params[k] for k in exit_keys if k in combined_params}

            df = FEATURES_BY_TF[tf]
            signals = generate_routed_signals(df, entry_params, strat_name)
            
            # FIX: Passed timeframe down correctly here
            trades_df, _ = run_backtest(
                timeframe=tf,
                df=df,
                signals=signals,
                entry_params=entry_params,
                exit_model=exit_model,
                exit_params=exit_params,
            )

            print(f"\n{'=' * 50}")
            print(f"  BEST RUN: {tf} | {strat_name.upper()} | {exit_model}")
            print(f"{'=' * 50}")
            generate_regime_report(trades_df)


  BEST RUN: M15 | TREND_PULLBACK | fixed_tp
      REGIME RESEARCH ATTRIBUTION SUMMARY      

REGIME: TREND
--------------------
Buys: 1098 | Sells: 1045
Net Profit: 39830.23 Cents
Profit Factor: 1.41
PnL Contribution: 100.0%

REGIME: SHOCK
--------------------
Buys: 0 | Sells: 0
Net Profit: 0.00 Cents
Profit Factor: N/A
PnL Contribution: 0.0%

REGIME: MR
--------------------
Buys: 0 | Sells: 0
Net Profit: 0.00 Cents
Profit Factor: N/A
PnL Contribution: 0.0%

  BEST RUN: M15 | VOLATILITY_EXPANSION | mr_exit
      REGIME RESEARCH ATTRIBUTION SUMMARY      

REGIME: TREND
--------------------
Buys: 0 | Sells: 0
Net Profit: 0.00 Cents
Profit Factor: N/A
PnL Contribution: 0.0%

REGIME: SHOCK
--------------------
Buys: 106 | Sells: 134
Net Profit: -2610.08 Cents
Profit Factor: 0.89
PnL Contribution: 100.0%

REGIME: MR
--------------------
Buys: 0 | Sells: 0
Net Profit: 0.00 Cents
Profit Factor: N/A
PnL Contribution: 0.0%

  BEST RUN: M5 | TREND_PULLBACK | fixed_tp
      REGIME RESEARCH ATTRI

In [12]:
MAX_DD_PCT = 25.0  # adjust cap (percent)

cols = [
    "timeframe", "strategy_name", "exit_model",
    "trade_count", "net_profit", "profit_per_trade",
    "max_drawdown", "robust_score", "profit_factor", "sharpe"
]

results_df = globals().get("all_results", None)
if results_df is None:
    results_df = globals().get("valid_results", None)
if results_df is None:
    raise NameError("all_results and valid_results are not defined. Run the backtest cell first.")

filtered = results_df[
    results_df["max_drawdown"] <= MAX_DD_PCT
]

display(
    filtered[cols]
    .sort_values(["timeframe", "robust_score"], ascending=[True, False])
    .reset_index(drop=True)
)

print("Remaining rows:", len(filtered))

,timeframe,strategy_name,exit_model,trade_count,net_profit,profit_per_trade,max_drawdown,robust_score,profit_factor,sharpe
0,M15,trend_pullback,fixed_tp,2778,34178.281412,12.303197,15.605280,8.404726,1.566285,7.816743
1,M15,trend_pullback,fixed_tp,2486,29843.372956,12.004575,16.543264,8.289623,1.547941,7.130244
2,M15,trend_pullback,fixed_tp,2396,31444.058355,13.123564,23.128466,8.235732,1.455073,6.521099
3,M15,trend_pullback,fixed_tp,2692,31674.638364,11.766210,15.687473,8.210226,1.533512,7.316974
4,M15,trend_pullback,fixed_tp,2328,29869.477804,12.830532,23.196186,8.130692,1.442676,6.271527
...,...,...,...,...,...,...,...,...,...,...
732,M5,trend_pullback,fixed_tp,8228,33970.019481,4.128588,22.938304,2.382328,1.431659,10.979535
733,M5,trend_pullback,fixed_tp_plus_mr,11584,32999.607145,2.848723,24.854268,2.365924,1.258349,7.985416
734,M5,trend_pullback,fixed_tp,11584,32843.531071,2.835250,24.766485,2.360463,1.256772,7.943514
735,M5,trend_pullback,fixed_tp_plus_mr,12758,51516.566957,4.037981,22.086674,2.351090,1.432977,13.996937


Remaining rows: 737


## Traceability Layer (Added)

The cells below were added to implement candidate-level observability, traceability, and reproducibility per the project goal. They are purely additive: no existing cell above (data loading, indicators, strategy definitions, the numba backtest engine, the grid/plateau search, or the CSV exports) was modified.

- Phase 1: `CandidateTrade` dataclass + deterministic `candidate_id`
- Phase 2: `CandidatePortfolio` built and exported for every winning parameter set
- Phase 3: `PipelineProfiler` recording candidate_id sets at each stage
- Phase 6: Signal Loss Report (per timeframe)
- Phase 7: `RejectionReason` enum + top rejection reasons per timeframe
- Phase 16: ML Readiness Score per exported parameter set (see the code comment on the documented proxy used for positive_label_ratio, since true TBM labels only exist in the Explorer notebook)

Run these cells after the existing export cells (Cells 0-12) have run, since they reuse `FEATURES_BY_TF`, `STRATEGIES`, `generate_routed_signals`, `run_backtest`, and `handoff`.

In [13]:
# ============================================================
# TRACEABILITY LAYER (ADDED) — Phase 1: Candidate Trade Object
# Pure addition. Does not modify strategy logic, backtest engine,
# CPCV, HMM, or XGBoost architecture.
# ============================================================
from dataclasses import dataclass, asdict
from enum import Enum
import hashlib


@dataclass
class CandidateTrade:
    candidate_id: str
    timeframe: str
    timestamp: pd.Timestamp
    direction: str  # "BUY" or "SELL"
    entry_price: float
    stop_price: float
    target_price: float
    strategy_name: str
    parameter_set_id: str
    hmm_state: int | None = None
    xgb_probability: float | None = None
    accepted: bool = False
    rejection_reason: str | None = None


def make_candidate_id(timeframe: str, timestamp, direction: str) -> str:
    """
    Deterministic candidate_id. This exact formula is reproduced in
    GoldRegimeX_Explorer.ipynb so IDs generated independently in both
    notebooks match for the same (timeframe, timestamp, direction).
    """
    key = f"{timeframe}_{timestamp}_{direction}"
    return hashlib.sha256(key.encode()).hexdigest()[:16]


def parameter_set_id(params: dict) -> str:
    """Deterministic id for a parameter set (stable across re-serialization)."""
    blob = json.dumps(params, sort_keys=True, default=str)
    return hashlib.sha256(blob.encode()).hexdigest()[:12]


class RejectionReason(str, Enum):
    LOW_PROBABILITY = "LOW_PROBABILITY"
    SESSION = "SESSION"
    ATR = "ATR"
    TBM = "TBM"
    DUPLICATE = "DUPLICATE"
    MAX_EXPOSURE = "MAX_EXPOSURE"
    INVALID_FEATURE = "INVALID_FEATURE"
    UNKNOWN = "UNKNOWN"


print("Phase 1 & 7: CandidateTrade, make_candidate_id, parameter_set_id, RejectionReason ready.")

Phase 1 & 7: CandidateTrade, make_candidate_id, parameter_set_id, RejectionReason ready.


In [14]:
# ============================================================
# TRACEABILITY LAYER (ADDED) — Phase 3: Pipeline Profiler
# ============================================================
class PipelineProfiler:
    """
    Reusable, append-only observability recorder.
    No stage may silently drop candidates: every stage transition is recorded
    with the exact candidate_id set that survived it.
    """

    def __init__(self):
        self.records = []  # list of dicts

    def record(self, timeframe, stage, candidate_ids, metadata=None):
        ids = set(candidate_ids)
        self.records.append(
            {
                "timeframe": timeframe,
                "stage": stage,
                "n_candidates": len(ids),
                "candidate_ids": ids,
                "metadata": metadata or {},
            }
        )

    def stage_ids(self, timeframe, stage):
        matches = [
            r["candidate_ids"]
            for r in self.records
            if r["timeframe"] == timeframe and r["stage"] == stage
        ]
        return matches[-1] if matches else set()

    def funnel_table(self, timeframe):
        rows = [
            {"stage": r["stage"], "n_candidates": r["n_candidates"], **r["metadata"]}
            for r in self.records
            if r["timeframe"] == timeframe
        ]
        return pd.DataFrame(rows)

    def signal_loss_report(self, timeframe, stage_order):
        """Compute candidate_ids lost between consecutive stages for one timeframe."""
        rows = []
        prev_ids = None
        prev_stage = None
        for stage in stage_order:
            cur_ids = self.stage_ids(timeframe, stage)
            if prev_ids is not None:
                lost = prev_ids - cur_ids
                rows.append(
                    {
                        "from_stage": prev_stage,
                        "to_stage": stage,
                        "stage_removed": stage,
                        "lost_count": len(lost),
                        "lost_candidate_ids": lost,
                    }
                )
            prev_ids, prev_stage = cur_ids, stage
        return pd.DataFrame(rows)


PIPELINE_STAGES = [
    "RAW_BARS",
    "FEATURE_ENGINEERING",
    "TREND_PULLBACK",
    "TBM",
    "LABEL_FILTER",
    "HMM",
    "XGBOOST",
    "PROBABILITY_FILTER",
    "BACKTEST",
    "RISK_MANAGER",
    "EXECUTED",
]

profiler = PipelineProfiler()
print("Phase 3: PipelineProfiler ready. Stages:", PIPELINE_STAGES)

Phase 3: PipelineProfiler ready. Stages: ['RAW_BARS', 'FEATURE_ENGINEERING', 'TREND_PULLBACK', 'TBM', 'LABEL_FILTER', 'HMM', 'XGBOOST', 'PROBABILITY_FILTER', 'BACKTEST', 'RISK_MANAGER', 'EXECUTED']


In [15]:
# ============================================================
# TRACEABILITY LAYER (ADDED) — Phase 1/2: Build CandidatePortfolio
# for each EXPORTED winning parameter set only (does not touch the
# grid search itself, so runtime/behaviour of Phases already run is
# unchanged -- this is purely additive instrumentation at export time).
# ============================================================


def compute_entry_stop_target(df, i, side, entry_params, is_m5):
    """
    Read-only re-derivation of the ENTRY pricing logic inside
    _run_backtest_numba, for a single bar, for observability only.
    Does NOT feed back into the compiled engine and does not change
    its behaviour in any way.
    Returns None if the engine would never have accepted this bar
    (ATR/spread gate failed).
    """
    atr_now = float(df["atr14"].iloc[i])
    close_px = float(df["close"].iloc[i])
    sp_points = float(df["spread"].iloc[i])
    effective_spread = (sp_points / 10.0) * PIP_SIZE_PRICE
    if not np.isfinite(atr_now) or atr_now <= 0.0 or effective_spread > (0.8 * atr_now):
        return None

    slippage_price = SLIPPAGE_PIPS * PIP_SIZE_PRICE
    guard_factor = (0.85 if side == 1 else 0.75) if is_m5 else 0.65
    intended_stop_dist = float(entry_params["atr_stop"]) * atr_now
    actual_stop_dist = intended_stop_dist * guard_factor
    tp_dist = float(entry_params["atr_target"]) * atr_now

    if side == 1:
        entry_px = close_px + effective_spread + slippage_price
        stop_px = entry_px - actual_stop_dist
        target_px = entry_px + tp_dist
    else:
        entry_px = close_px - slippage_price
        stop_px = entry_px + actual_stop_dist + effective_spread
        target_px = entry_px - tp_dist
    return entry_px, stop_px, target_px


def build_position_active_mask(df, trades_df):
    """Approximate 'already in a position' mask from an executed trades_df."""
    n = len(df)
    mask = np.zeros(n, dtype=bool)
    if trades_df is None or trades_df.empty:
        return mask
    pos = {ts: i for i, ts in enumerate(df.index)}
    for _, row in trades_df.iterrows():
        e = pos.get(row["entry_time"])
        x = pos.get(row["exit_time"])
        if e is not None and x is not None:
            mask[e : x + 1] = True
    return mask


def build_candidate_portfolio(timeframe, strategy_name, exit_model, params):
    """
    Phase 1/2: Build a CandidatePortfolio for one exported winning parameter set.
    Only called for exported winners (top_n rows), never during grid search,
    so no optimisation behaviour or runtime is changed.
    """
    df = FEATURES_BY_TF[timeframe]
    is_m5 = timeframe == "M5"

    entry_params = {k: params[k] for k in STRATEGIES[strategy_name].param_cols if k in params}
    signals = generate_routed_signals(df, entry_params, strategy_name)

    exit_params = {
        k: params[k] for k in ("leg_a_atr_target", "time_stop_minutes", "trail_mult") if k in params
    }
    trades_df, metrics = run_backtest(
        timeframe=timeframe,
        df=df,
        signals=signals,
        entry_params=entry_params,
        exit_model=exit_model,
        exit_params=exit_params,
    )

    executed_entry_times = set(trades_df["entry_time"]) if not trades_df.empty else set()
    position_active = build_position_active_mask(df, trades_df)
    pset_id = parameter_set_id(params)

    candidates = []
    sig_arr = signals.to_numpy()
    nonzero_positions = np.flatnonzero(sig_arr != 0)

    # Only Trend Pullback candidates get a CandidateTrade object immediately
    # "after the strategy generates a candidate setup" per Phase 1. Other
    # strategies (e.g. volatility_expansion) are tagged the same way for
    # portfolio completeness, but the spec's traceability focus is Trend Pullback.
    for i in nonzero_positions:
        ts = df.index[i]
        side = int(sig_arr[i])
        direction = "BUY" if side == 1 else "SELL"
        cid = make_candidate_id(timeframe, ts, direction)

        priced = compute_entry_stop_target(df, i, side, entry_params, is_m5)
        accepted = ts in executed_entry_times

        if priced is None:
            entry_price = stop_price = target_price = float("nan")
            rejection_reason = None if accepted else RejectionReason.ATR.value
        else:
            entry_price, stop_price, target_price = priced
            if accepted:
                rejection_reason = None
            elif position_active[i]:
                rejection_reason = RejectionReason.DUPLICATE.value
            else:
                rejection_reason = RejectionReason.UNKNOWN.value

        candidates.append(
            CandidateTrade(
                candidate_id=cid,
                timeframe=timeframe,
                timestamp=ts,
                direction=direction,
                entry_price=float(entry_price),
                stop_price=float(stop_price),
                target_price=float(target_price),
                strategy_name=strategy_name,
                parameter_set_id=pset_id,
                hmm_state=None,
                xgb_probability=None,
                accepted=bool(accepted),
                rejection_reason=rejection_reason,
            )
        )

    all_ids = [c.candidate_id for c in candidates]
    executed_ids = [c.candidate_id for c in candidates if c.accepted]
    meta = {"strategy": strategy_name, "parameter_set_id": pset_id}
    profiler.record(timeframe, "TREND_PULLBACK", all_ids, metadata=meta)
    profiler.record(timeframe, "BACKTEST", all_ids, metadata=meta)
    profiler.record(timeframe, "EXECUTED", executed_ids, metadata=meta)

    portfolio = {
        "timeframe": timeframe,
        "strategy": strategy_name,
        "exit_model": exit_model,
        "parameter_set": params,
        "parameter_set_id": pset_id,
        "candidate_trades": [asdict(c) for c in candidates],
        "summary_metrics": metrics,
    }
    return portfolio


print("Phase 1/2: build_candidate_portfolio ready.")

Phase 1/2: build_candidate_portfolio ready.


In [16]:
# ============================================================
# TRACEABILITY LAYER (ADDED) — Phase 16: ML Readiness Score
# NOTE: true 'positive_label_ratio' requires TBM labels, which are computed
# in the Explorer notebook (not here). This score therefore proxies
# positive_label_ratio with the execution-acceptance rate of generated
# candidates, documented explicitly so it is not mistaken for the TBM ratio.
# ============================================================


def compute_ml_readiness_score(timeframe, portfolio):
    candidates = portfolio["candidate_trades"]
    n = len(candidates)
    if n == 0:
        return {
            "timeframe": timeframe,
            "candidate_count": 0,
            "ml_readiness_score": 0.0,
            "temporal_diversity": 0.0,
            "regime_diversity": 0.0,
            "class_balance_proxy": 0.0,
            "oos_balance_proxy": 0.0,
            "positive_label_ratio_proxy": 0.0,
        }

    ts_series = pd.to_datetime([c["timestamp"] for c in candidates])
    months_covered = ts_series.to_period("M").nunique()
    total_months = max(1, (ts_series.max().to_period("M") - ts_series.min().to_period("M")).n + 1)
    temporal_diversity = months_covered / total_months

    directions = pd.Series([c["direction"] for c in candidates])
    dir_counts = directions.value_counts(normalize=True)
    class_balance_proxy = 1.0 - abs(dir_counts.get("BUY", 0.0) - dir_counts.get("SELL", 0.0))

    df_tf = FEATURES_BY_TF[timeframe]
    regimes = df_tf["regime_str"].reindex(ts_series)
    regime_counts = regimes.value_counts(normalize=True, dropna=True)
    regime_diversity = float(1.0 - (regime_counts.pow(2).sum())) if len(regime_counts) else 0.0

    split_point = ts_series.min() + (ts_series.max() - ts_series.min()) * 0.8
    is_count = int((ts_series <= split_point).sum())
    oos_count = n - is_count
    oos_balance_proxy = (
        min(is_count, oos_count) / max(is_count, oos_count) if max(is_count, oos_count) > 0 else 0.0
    )

    candidate_count_score = min(1.0, n / 500.0)
    positive_label_ratio_proxy = float(pd.Series([c["accepted"] for c in candidates]).mean())

    weights = {
        "candidate_count": 0.20,
        "positive_label_ratio": 0.20,
        "temporal_diversity": 0.15,
        "regime_diversity": 0.15,
        "class_balance": 0.15,
        "oos_balance": 0.15,
    }
    score = (
        weights["candidate_count"] * candidate_count_score
        + weights["positive_label_ratio"] * positive_label_ratio_proxy
        + weights["temporal_diversity"] * temporal_diversity
        + weights["regime_diversity"] * regime_diversity
        + weights["class_balance"] * class_balance_proxy
        + weights["oos_balance"] * oos_balance_proxy
    )

    return {
        "timeframe": timeframe,
        "candidate_count": n,
        "candidate_count_score": candidate_count_score,
        "positive_label_ratio_proxy": positive_label_ratio_proxy,
        "temporal_diversity": temporal_diversity,
        "regime_diversity": regime_diversity,
        "class_balance_proxy": class_balance_proxy,
        "oos_balance_proxy": oos_balance_proxy,
        "ml_readiness_score": float(score),
    }


print("Phase 16: compute_ml_readiness_score ready (positive_label_ratio is a documented proxy).")

Phase 16: compute_ml_readiness_score ready (positive_label_ratio is a documented proxy).


In [17]:
# ============================================================
# TRACEABILITY LAYER (ADDED) — Phase 2/6/7/16: Assemble + Export
# Extends (does not replace) the existing 'winners_for_explorer.csv' export.
# ============================================================
os.makedirs("reports", exist_ok=True)

candidate_portfolios = []
readiness_rows = []
flat_candidate_rows = []

for _, row in handoff.iterrows():
    tf = row["timeframe"]
    strategy_name = row["strategy_name"]
    exit_model = row["exit_model"]
    params = json.loads(row["parameter_set"])

    portfolio = build_candidate_portfolio(tf, strategy_name, exit_model, params)
    candidate_portfolios.append(portfolio)

    readiness = compute_ml_readiness_score(tf, portfolio)
    readiness["strategy_name"] = strategy_name
    readiness["exit_model"] = exit_model
    readiness["parameter_set_id"] = portfolio["parameter_set_id"]
    readiness_rows.append(readiness)

    for c in portfolio["candidate_trades"]:
        flat_candidate_rows.append(
            {
                **c,
                "strategy_name": strategy_name,
                "exit_model": exit_model,
                "parameter_set_id": portfolio["parameter_set_id"],
            }
        )

with open("reports/candidate_portfolios.json", "w") as f:
    json.dump(candidate_portfolios, f, default=str, indent=2)
print("Saved: reports/candidate_portfolios.json")

candidate_trades_export = pd.DataFrame(flat_candidate_rows)
candidate_trades_export.to_csv("reports/candidate_trades_export.csv", index=False)
print("Saved: reports/candidate_trades_export.csv  (%d candidate rows)" % len(candidate_trades_export))
if not candidate_trades_export.empty:
    assert "candidate_id" in candidate_trades_export.columns

ml_readiness_df = pd.DataFrame(readiness_rows)
ml_readiness_df.to_csv("reports/ml_readiness_scores.csv", index=False)
print("Saved: reports/ml_readiness_scores.csv")
display(ml_readiness_df)

# --- Phase 6: Signal Loss Report per timeframe (Trend Pullback -> Executed) ---
print("\nSignal Loss Report")
for tf in TIMEFRAMES:
    loss_df = profiler.signal_loss_report(tf, ["TREND_PULLBACK", "BACKTEST", "EXECUTED"])
    print(f"\n{tf}:")
    if loss_df.empty:
        print("  (no stages recorded)")
    else:
        display(loss_df[["from_stage", "to_stage", "lost_count"]])
    loss_df.drop(columns=["lost_candidate_ids"], errors="ignore").to_csv(
        f"reports/signal_loss_report_{tf}_strategy_tester.csv", index=False
    )

# --- Phase 7: Top rejection reasons per timeframe ---
print("\nTop rejection reasons per timeframe")
for tf in TIMEFRAMES:
    tf_rows = [r for r in flat_candidate_rows if r["timeframe"] == tf and not r["accepted"]]
    if not tf_rows:
        print(f"  {tf}: no rejected candidates")
        continue
    reasons = pd.Series([r["rejection_reason"] for r in tf_rows]).value_counts()
    print(f"\n  {tf}:")
    print(reasons.to_string())

print("\nPhase 1/2/3/6/7/16 traceability exports complete.")

Saved: reports/candidate_portfolios.json
Saved: reports/candidate_trades_export.csv  (7642560 candidate rows)
Saved: reports/ml_readiness_scores.csv


,timeframe,candidate_count,candidate_count_score,positive_label_ratio_proxy,temporal_diversity,regime_diversity,class_balance_proxy,oos_balance_proxy,ml_readiness_score,strategy_name,exit_model,parameter_set_id
0,M15,2533,1.0,0.659692,1.0,0.0,0.957758,0.257696,0.664256,trend_pullback,fixed_tp,d01afb4fd74e
1,M15,2247,1.0,0.667112,1.0,0.0,0.958611,0.258119,0.665932,trend_pullback,fixed_tp,d4727b31a3e2
2,M15,2533,1.0,0.508883,1.0,0.0,0.957758,0.257696,0.634095,trend_pullback,fixed_tp,9fda9f9349e7
3,M15,2449,1.0,0.659453,1.0,0.0,0.956309,0.257187,0.663915,trend_pullback,fixed_tp,d0e0e5b3e49a
4,M15,2449,1.0,0.510821,1.0,0.0,0.956309,0.257187,0.634188,trend_pullback,fixed_tp,46596cff594d
...,...,...,...,...,...,...,...,...,...,...,...,...
752,M15,1976,1.0,0.508603,1.0,0.0,0.959514,0.252218,0.633480,trend_pullback,fixed_tp,7146a414432f
753,M15,2102,1.0,0.440533,1.0,0.0,0.956232,0.256426,0.620005,trend_pullback,fixed_tp,c694e28d1dcc
754,M15,2962,1.0,0.479406,1.0,0.0,0.992573,0.250317,0.632315,trend_pullback,fixed_tp,b2f41d0271fe
755,M15,3209,1.0,0.474603,1.0,0.0,0.988470,0.252048,0.630998,trend_pullback,fixed_tp,1abe1d6423ba



Signal Loss Report

M15:


,from_stage,to_stage,lost_count
0,TREND_PULLBACK,BACKTEST,0
1,BACKTEST,EXECUTED,1700



M5:


,from_stage,to_stage,lost_count
0,TREND_PULLBACK,BACKTEST,0
1,BACKTEST,EXECUTED,6091



Top rejection reasons per timeframe

  M15:
DUPLICATE    631636

  M5:
DUPLICATE    3080761
ATR            41634

Phase 1/2/3/6/7/16 traceability exports complete.


## Pipeline Verification & Certification (Added)

Additive cells implementing Phases 1-17 for the Strategy Tester's scope (candidate generation + session filtering + shared manifest). The ML training phases (2 dataset integrity, 3 train/OOS leakage, 4 feature leakage, 5-7 evaluation/alignment/model identity, 11 calibration, 12 threshold, 13 root cause, 14 candidate integrity, 15 full waterfall, 16 certification) live in the Explorer notebook because the ML pipeline itself is defined there.

Requires `pipeline_verification.py` and `shared/session_filter.py` in the same working directory. Gated on `VERIFY_PIPELINE = True`.

In [18]:
# ============================================================
# PIPELINE VERIFICATION (ADDED) - Global flag + imports
# Phases 1, 8. Purely additive - all subsequent verification cells
# are gated on VERIFY_PIPELINE so runtime behaviour is unchanged
# when the flag is False.
# ============================================================
import sys, os, json, hashlib
from pathlib import Path

VERIFY_PIPELINE = True  # set False to skip all verification below with zero overhead

if VERIFY_PIPELINE:
    # Make the local pipeline_verification.py and shared/session_filter.py importable.
    for _p in (".", str(Path.cwd())):
        if _p not in sys.path:
            sys.path.insert(0, _p)
    from shared.session_filter import SessionFilter, run_session_filter_test_suite
    import pipeline_verification as pv
    from pipeline_verification import (
        DatasetIntegrityVerifier,
        FeatureLeakageVerifier,
        EvaluationVerifier,
        PredictionAlignmentVerifier,
        SessionVerifier,
        ThresholdVerifier,
        CandidateIntegrityVerifier,
        PipelineCertification,
        RootCauseAnalyzerV2,
        build_pipeline_waterfall,
        build_manifest,
    )
    session_filter = SessionFilter()
    certification = PipelineCertification()
    print("VERIFY_PIPELINE=True. SessionFilter hash:", session_filter.version_hash())
else:
    print("VERIFY_PIPELINE=False. Skipping all verification cells.")

VERIFY_PIPELINE=True. SessionFilter hash: 2e32a85f9217264a


In [19]:
# ============================================================
# PIPELINE VERIFICATION (ADDED) - Phase 9: Session Filter Test Suite
# ============================================================
if VERIFY_PIPELINE:
    session_test_df, session_test_warnings = run_session_filter_test_suite(session_filter)
    session_status = "PASS" if (session_test_df["result"] == "PASS").all() else "FAIL"
    certification.record(pv.VerificationResult(
        "Session Consistency", session_status,
        {"tests": session_test_df, "warnings": session_test_warnings},
    ))

==========================  SESSION FILTER TEST REPORT  ==========================
                                 test        timestamp    filter  expected  actual detected_session  weekend result
          London open (Mon 07:00 UTC) 2025-01-06 07:00    London      True    True           LONDON    False   PASS
         London close (Mon 15:59 UTC) 2025-01-06 15:59    London      True    True          OVERLAP    False   PASS
    Post-London close (Mon 16:00 UTC) 2025-01-06 16:00    London     False   False         NEW_YORK    False   PASS
     New York overlap (Mon 14:00 UTC) 2025-01-06 14:00 London_NY      True    True          OVERLAP    False   PASS
                 Asia (Mon 03:00 UTC) 2025-01-06 03:00 London_NY     False   False             ASIA    False   PASS
         Friday close (Fri 20:59 UTC) 2025-01-10 20:59        NY      True    True         NEW_YORK    False   PASS
          Sunday open (Sun 22:00 UTC) 2025-01-05 22:00 London_NY     False   False             ASIA     T

In [20]:
# ============================================================
# PIPELINE VERIFICATION (ADDED) - Phase 10: Session Audit Report
# Runs against the candidate portfolios already exported by earlier
# Traceability Layer cells (reports/candidate_trades_export.csv).
# Does not modify Trend Pullback / session logic.
# ============================================================
if VERIFY_PIPELINE:
    candidates_path = Path("reports/candidate_trades_export.csv")
    if candidates_path.exists():
        import pandas as pd
        exported_df = pd.read_csv(candidates_path, parse_dates=["timestamp"])
        verifier = SessionVerifier(session_filter)
        for tf in sorted(exported_df["timeframe"].unique()):
            tf_rows = exported_df[exported_df["timeframe"] == tf].head(500)
            candidates = tf_rows.to_dict(orient="records")
            # Use London_NY as the strictest common filter for auditing.
            result = verifier.audit(candidates, session_filter_value="London_NY")
            print("Session audit above for timeframe:", tf)
    else:
        print("No reports/candidate_trades_export.csv yet; run the earlier Traceability export cell first.")

                        SESSION AUDIT REPORT
    candidate_id           timestamp            utc_time           broker_time detected_session  accepted
15b689732303106f 2021-06-02 14:45:00 2021-06-02 14:45:00 2021-06-02 17:45 EEST          OVERLAP      True
a416f21ca5a6833b 2021-06-02 15:00:00 2021-06-02 15:00:00 2021-06-02 18:00 EEST          OVERLAP      True
eb2c3c458eb1ad51 2021-06-02 15:15:00 2021-06-02 15:15:00 2021-06-02 18:15 EEST          OVERLAP      True
b35a0e3b7cb8860c 2021-06-02 15:30:00 2021-06-02 15:30:00 2021-06-02 18:30 EEST          OVERLAP      True
a78a31272317b9f4 2021-06-02 15:45:00 2021-06-02 15:45:00 2021-06-02 18:45 EEST          OVERLAP      True
9b52faa4c32918b8 2021-06-02 16:00:00 2021-06-02 16:00:00 2021-06-02 19:00 EEST         NEW_YORK      True
4bee8e47101e6bb0 2021-06-02 16:15:00 2021-06-02 16:15:00 2021-06-02 19:15 EEST         NEW_YORK      True
b2383c5db48d51b5 2021-06-02 20:15:00 2021-06-02 20:15:00 2021-06-02 23:15 EEST         NEW_YORK      True
8

In [21]:
# ============================================================
# PIPELINE VERIFICATION (ADDED) - Phase 15 (ST portion): waterfall of
# candidate GENERATION (Strategy Tester only observes up to signal-
# generation and session filtering). HMM/TBM/XGBoost stages live in
# the Explorer notebook, which produces the full seven-stage waterfall.
# ============================================================
if VERIFY_PIPELINE:
    if candidates_path.exists():
        for tf in sorted(exported_df["timeframe"].unique()):
            tf_rows = exported_df[exported_df["timeframe"] == tf]
            generated = len(tf_rows)
            session_pass = int(sum(
                session_filter.passes(t, "London_NY") for t in tf_rows["timestamp"]
            ))
            stage_counts = {
                "Generated": generated,
                "Session": session_pass,
            }
            print(f"Waterfall (Strategy Tester scope) - {tf}")
            build_pipeline_waterfall(stage_counts)

Waterfall (Strategy Tester scope) - M15
                         PIPELINE WATERFALL
    stage  remaining  lost  cumulative_survival_pct  marginal_loss_pct
Generated    1533201     0                    100.0                0.0
  Session    1509784 23417                     98.5                1.5
Waterfall (Strategy Tester scope) - M5
                         PIPELINE WATERFALL
    stage  remaining   lost  cumulative_survival_pct  marginal_loss_pct
Generated    6109359      0                    100.0                0.0
  Session    5368699 740660                     87.9               12.1


In [22]:
# ============================================================
# PIPELINE VERIFICATION (ADDED) - Phase 17: shared manifest export
# The Explorer reads reports/pipeline_manifest_strategy_tester.json
# and cross-checks it against its own manifest.
# ============================================================
if VERIFY_PIPELINE:
    import pandas as pd
    os.makedirs("reports", exist_ok=True)
    if candidates_path.exists():
        # Extract the union of columns already used by exported candidates as the
        # "feature set" fingerprint (kept generic - actual ML features are only
        # engineered inside the Explorer notebook).
        strategy_version = "trend_pullback@v1"
        pipeline_version = "gxr-verify@1.0.0"
        manifest = build_manifest(
            pipeline_version=pipeline_version,
            strategy_version=strategy_version,
            feature_columns=["timestamp", "direction", "entry_price", "stop_price", "target_price"],
            session_filter_hash=session_filter.version_hash(),
            candidate_ids=exported_df["candidate_id"].tolist(),
            train_hash="",  # Strategy Tester does not train ML models
            oos_hash="",
            extra={"per_timeframe_counts": exported_df["timeframe"].value_counts().to_dict()},
        )
        with open("reports/pipeline_manifest_strategy_tester.json", "w") as f:
            json.dump(manifest, f, indent=2, default=str)
        print("Manifest exported:", manifest)

Manifest exported: {'pipeline_version': 'gxr-verify@1.0.0', 'strategy_version': 'trend_pullback@v1', 'feature_set_hash': '8d23ba410e018cd3', 'session_filter_hash': '2e32a85f9217264a', 'candidate_count': 7642560, 'candidate_ids_hash': '0efc8e74c1ca130f', 'train_hash': '', 'oos_hash': '', 'per_timeframe_counts': {'M5': 6109359, 'M15': 1533201}}


In [23]:
# ============================================================
# PIPELINE VERIFICATION (ADDED) - Phase 16: mini certification for the
# Strategy Tester's scope. Full certification (all 10 categories) is
# produced by the Explorer notebook, which sees the ML stages too.
# ============================================================
if VERIFY_PIPELINE:
    cert = certification.certify()
    with open("reports/pipeline_certification_strategy_tester.json", "w") as f:
        json.dump({"overall": cert["overall"], "results": cert["results"]}, f, indent=2, default=str)
    print("Strategy Tester certification saved to reports/pipeline_certification_strategy_tester.json")

                       PIPELINE CERTIFICATION
               category  status
   Train/OOS Separation SKIPPED
        Feature Leakage SKIPPED
   Evaluation Integrity SKIPPED
   Prediction Alignment SKIPPED
    Session Consistency    PASS
    Candidate Integrity SKIPPED
            Model Reuse SKIPPED
 Threshold Verification SKIPPED
Probability Calibration SKIPPED
    Root Cause Analysis SKIPPED
      Dataset Integrity SKIPPED
Overall Status: PASS
Strategy Tester certification saved to reports/pipeline_certification_strategy_tester.json
